# Model Evaluation & Benchmark: Fine-Tuned Multi-Class RF-DETR vs. Old Model Endpoint
### Strict Benchmark Architecture: COCO Annotations as Ground Truth

This notebook implements a 3-way evaluation benchmark:
1. **Ground Truth (COCO Test Annotations)**:
   - Extracted directly from `test/_annotations.coco.json`.
   - Contains verified bounding boxes and labels for all three categories:
     - `location_tag` (Standard white location tags)
     - `Blue_aisle` (Blue aisle header signs)
     - `blue_bay` (Blue bay shelf location tags)
2. **Old Model Inference (Production REST API Endpoint)**:
   - Detections retrieved by querying the production endpoint (`https://prod-itemrecognitionservice.cld.samsclub.com/v3/location_tag`).
   - Evaluated against the **COCO Ground Truth** to measure white tag accuracy and behavior on blue tags.
3. **New Model Inference (RF-DETR Multi-Class)**:
   - Detections generated locally by the fine-tuned RF-DETR model (`best_model_full_data.pth`).
   - Evaluated against the **COCO Ground Truth** across all 3 classes.

---

### Core Comparison Protocol:
- **Unified Ground Truth**: Both models are scored against the **exact same COCO ground-truth annotations** on the held-out test split.
- **Head-to-Head White Tag Evaluation**: Compares Old Model vs. RF-DETR specifically on `location_tag` (white tags).
- **Class-Agnostic Tag Localization**: Compares how reliably each model detects ANY physical tag in the aisle ($	ext{IoU} \ge 0.50$).
- **Multi-Class Capability**: Evaluates RF-DETR's ability to classify `Blue_aisle` and `blue_bay` vs. the Old Model's limitations.


In [ ]:
# STEP 0: Environment Dependencies & Compatibility Setup
# Install required dependencies for evaluation, metrics, and visualization
%pip install -q torchmetrics supervision pycocotools pandas requests urllib3 Pillow opencv-python-headless matplotlib tabulate

import sys
import subprocess
print(f"Python executable: {sys.executable}")
print("Dependencies verification complete.")


In [ ]:
# CELL 1: Imports, Multiprocessing Configuration & Logger Setup
import os
import sys
import json
import time
import math
import copy
import uuid
import base64
import logging
import random
import shutil
import warnings
import re
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

# Suppress warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)

# Disable insecure HTTPS warnings for the internal endpoint
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configure progress bar
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

import cv2
import torch
import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageDraw, ImageFont
import torchvision.transforms.functional as TF

try:
    import supervision as sv
except ImportError:
    sv = None

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
except ImportError:
    MeanAveragePrecision = None

# Multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

# Custom Logger with auto-flush
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("evaluate_compare_models")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s"))
logger.addHandler(console_handler)

# GPU / Hardware Info
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Execution Device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU Name: {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM Allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
else:
    logger.info("Running on CPU mode.")


In [ ]:
# CELL 2: Master Configuration & Directory Paths
# Configuration for multi-class RF-DETR model architecture & inference
logger.info("=" * 80)
logger.info("[CELL 2] Master Configuration: Categories, Dataset Paths & Endpoint Settings")
logger.info("=" * 80)

# -------------------------------------------------------------------------
# 1. Pipeline Directory Structure & Test Split Paths
# -------------------------------------------------------------------------
REPO_ROOT = Path(".")
PIPELINE_NAME = "multi_class_train_rfdetr"
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME

# Locate Test Annotations File (_annotations.coco.json)
CANDIDATE_TEST_ANNS = [
    PIPELINE_DIR / "dataset_full_data" / "test" / "_annotations.coco.json",
    PIPELINE_DIR / "dataset_sample_1000" / "test" / "_annotations.coco.json",
    PIPELINE_DIR / "dataset" / "test" / "_annotations.coco.json",
    REPO_ROOT / "coco_files" / "test" / "_annotations.coco.json",
    REPO_ROOT / "test" / "_annotations.coco.json",
    REPO_ROOT / "test" / "test_annotations.json",
]

TEST_ANN_PATH = None
for p in CANDIDATE_TEST_ANNS:
    if p.exists() and p.stat().st_size > 0:
        TEST_ANN_PATH = p
        break

if TEST_ANN_PATH is None:
    TEST_ANN_PATH = PIPELINE_DIR / "dataset_full_data" / "test" / "_annotations.coco.json"

# Candidate Image Directories for Test Images
CANDIDATE_IMAGE_DIRS = [
    TEST_ANN_PATH.parent / "images",
    TEST_ANN_PATH.parent,
    PIPELINE_DIR / "images",
    REPO_ROOT / "images",
    REPO_ROOT / "coco_files" / "images",
]

TEST_IMAGES_DIR = TEST_ANN_PATH.parent / "images"
for d in CANDIDATE_IMAGE_DIRS:
    if d.exists() and any(d.iterdir()):
        TEST_IMAGES_DIR = d
        break

logger.info(f"Selected Test Annotations Path: {TEST_ANN_PATH}")
logger.info(f"Selected Test Images Directory:   {TEST_IMAGES_DIR}")

# -------------------------------------------------------------------------
# 2. Model Architecture & Hyperparameters
# -------------------------------------------------------------------------
# Target categories for New Multi-Class RF-DETR Model:
# Index 0: blue_aisle
# Index 1: blue_bay
# Index 2: location_tag
NEW_MODEL_CLASSES = ["blue_aisle", "blue_bay", "location_tag"]
TARGET_CATEGORIES = NEW_MODEL_CLASSES
WHITE_TAG_CATEGORY = "location_tag"

CANDIDATE_RFDETR_CHECKPOINTS = [
    PIPELINE_DIR / "model" / "best_model_full_data.pth",
    PIPELINE_DIR / "model" / "best_model_sample_1000.pth",
    PIPELINE_DIR / "runs" / "rf_detr_full_data" / "checkpoints" / "best_loss.pth",
    PIPELINE_DIR / "runs" / "rf_detr_sample_1000" / "checkpoints" / "best_loss.pth",
    PIPELINE_DIR / "model" / "latest_checkpoint.pth",
    REPO_ROOT / "best_model.pth",
]

RFDETR_CHECKPOINT_PATH = None
for ckpt in CANDIDATE_RFDETR_CHECKPOINTS:
    if ckpt.exists() and ckpt.stat().st_size > 1024 * 1024:
        RFDETR_CHECKPOINT_PATH = ckpt
        break

if RFDETR_CHECKPOINT_PATH is None:
    RFDETR_CHECKPOINT_PATH = PIPELINE_DIR / "model" / "best_model_full_data.pth"

MODEL_SIZE = "base"
TARGET_RESOLUTION = 1008
DIVISOR = 56 if MODEL_SIZE == "base" else 32
RESOLUTION = max(round(TARGET_RESOLUTION / DIVISOR) * DIVISOR, DIVISOR)

# Parameterized Confidence Threshold for New Model (e.g., 0.15, 0.25, 0.45, 0.50)
CONFIDENCE_THRESHOLD = 0.45
IOU_THRESHOLD = 0.50

# -------------------------------------------------------------------------
# 2B. High-Throughput Inference & Batching Settings
# Boosts RF-DETR evaluation from ~2 images/sec to 6-12+ images/sec:
# 1) Mini-batch processing (amortizes GPU kernel launch & saturates Tensor Cores)
# 2) PyTorch DataLoader with asynchronous multi-worker prefetching (hides disk I/O & PIL resize)
# 3) torch.inference_mode() + cuDNN autotuning
# -------------------------------------------------------------------------
EVAL_BATCH_SIZE = 4       # Mini-batch size (e.g. 4 or 8 for 6-12+ FPS; 2 for low VRAM)
NUM_EVAL_WORKERS = 4 if torch.cuda.is_available() else 0  # Background CPU workers for asynchronous pre-fetching

logger.info(f"RF-DETR Target Classes:          {NEW_MODEL_CLASSES}")
logger.info(f"RF-DETR Checkpoint Path:         {RFDETR_CHECKPOINT_PATH} (Exists: {RFDETR_CHECKPOINT_PATH.exists()})")
logger.info(f"RF-DETR Architecture:            RF-DETR {MODEL_SIZE.capitalize()} | Resolution: {RESOLUTION}x{RESOLUTION}")
logger.info(f"Configured Confidence Threshold: {CONFIDENCE_THRESHOLD} (Parameterized)")
logger.info(f"Configured IoU Threshold:        {IOU_THRESHOLD}")
logger.info(f"High-Speed Evaluation Batch Size:{EVAL_BATCH_SIZE} | Workers: {NUM_EVAL_WORKERS}")

# -------------------------------------------------------------------------
# 3. Old Model REST API Configuration
# -------------------------------------------------------------------------
API_URL = "https://prod-itemrecognitionservice.cld.samsclub.com/v3/location_tag"
CLUB_ID = "4822"
SAMPLE_SIZE = None      # None = process all test images
NUM_API_WORKERS = 16    # Parallel worker threads for endpoint calls
API_TIMEOUT = 60
API_OUTPUT_CSV = REPO_ROOT / "location_tag_detections.csv"

# Output Directory for Evaluation Artifacts
EVAL_OUTPUT_DIR = PIPELINE_DIR / "evaluation_compare_endpoint"
PREVIEWS_DIR = EVAL_OUTPUT_DIR / "previews"
CHARTS_DIR = EVAL_OUTPUT_DIR / "charts"
for p in [EVAL_OUTPUT_DIR, PREVIEWS_DIR, CHARTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

logger.info(f"Evaluation Outputs Directory:    {EVAL_OUTPUT_DIR}")
logger.info("Configuration loaded successfully.")


In [ ]:
# CELL 3: Test Dataset Loading, Dynamic Category Discovery & Image Downloader
# [RULE] If anything is already downloaded, SKIP IT completely!
logger.info("=" * 80)
logger.info("[CELL 3] Loading Test Annotations & Dynamic Category Discovery from COCO")
logger.info("=" * 80)

# Verify or locate COCO annotation file
if not TEST_ANN_PATH.exists():
    logger.warning(f"Test annotation file not found at {TEST_ANN_PATH}!")
    coco_files_dir = REPO_ROOT / "coco_files"
    if coco_files_dir.exists():
        found_jsons = list(coco_files_dir.glob("*.json"))
        logger.info(f"Searching in {coco_files_dir}: found {len(found_jsons)} json files.")
        for jf in found_jsons:
            if "test" in jf.name.lower():
                TEST_ANN_PATH = jf
                break
        if not TEST_ANN_PATH.exists() and found_jsons:
            TEST_ANN_PATH = found_jsons[0]

assert TEST_ANN_PATH.exists(), f"Fatal: COCO test annotation file not found at {TEST_ANN_PATH}."

with open(TEST_ANN_PATH, "r", encoding="utf-8") as f:
    test_coco_data = json.load(f)

raw_images = test_coco_data.get("images", [])
raw_annotations = test_coco_data.get("annotations", [])
raw_categories = test_coco_data.get("categories", [])

logger.info(f"COCO file loaded: {len(raw_images)} images, {len(raw_annotations)} annotations, {len(raw_categories)} categories.")

# =========================================================================
# MULTI-CLASS TARGET CATEGORIES & COCO CATEGORY NORMALIZATION
# Classes:
#   Index 0: blue_aisle
#   Index 1: blue_bay
#   Index 2: location_tag
# =========================================================================
class_names = NEW_MODEL_CLASSES
TARGET_CATEGORIES = NEW_MODEL_CLASSES
WHITE_TAG_CATEGORY = "location_tag"
white_tag_idx = NEW_MODEL_CLASSES.index("location_tag")

def normalize_tag_name(raw_name: str) -> str:
    """Normalizes category names across datasets (e.g. 'Blue_aisle' -> 'blue_aisle')."""
    n = raw_name.strip().lower().replace(" ", "_")
    if "aisle" in n: return "blue_aisle"
    if "bay" in n or "tag" in n and "loc" not in n and "white" not in n: return "blue_bay"
    return "location_tag"

# Map COCO category IDs directly to normalized target classes
sorted_cats = sorted(raw_categories, key=lambda c: c["id"])
cat_id_to_cname = {}
cat_id_to_idx = {}
for i, c in enumerate(sorted_cats):
    norm_name = normalize_tag_name(c["name"])
    # If COCO names already match, preserve order; else map to canonical NEW_MODEL_CLASSES
    if norm_name in NEW_MODEL_CLASSES:
        idx = NEW_MODEL_CLASSES.index(norm_name)
    else:
        idx = min(i, len(NEW_MODEL_CLASSES) - 1)
        norm_name = NEW_MODEL_CLASSES[idx]
    cat_id_to_cname[c["id"]] = norm_name
    cat_id_to_idx[c["id"]] = idx

idx_to_class_name = {i: cname for i, cname in enumerate(NEW_MODEL_CLASSES)}
class_name_to_idx = {cname: i for i, cname in enumerate(NEW_MODEL_CLASSES)}

logger.info("=" * 80)
logger.info("NEW MODEL TARGET CATEGORIES:")
for i, name in enumerate(NEW_MODEL_CLASSES):
    is_w = " [WHITE TAG BASELINE]" if name == WHITE_TAG_CATEGORY else ""
    logger.info(f"   - Model Output Index {i} -> '{name}'{is_w}")
logger.info("COCO Annotation Category Mapping:")
for c in sorted_cats:
    logger.info(f"   - COCO Category ID {c['id']}: '{c['name']}' -> Mapped to '{cat_id_to_cname[c['id']]}'")
logger.info("=" * 80)

annotations_by_image_id = defaultdict(list)
for ann in raw_annotations:
    annotations_by_image_id[ann["image_id"]].append(ann)

# -------------------------------------------------------------------------
# COMPREHENSIVE LOCAL CACHE SCANNER: Search all possible image locations
# -------------------------------------------------------------------------
TEST_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_SEARCH_DIRS = [
    TEST_IMAGES_DIR,
    TEST_ANN_PATH.parent / "images",
    TEST_ANN_PATH.parent,
    PIPELINE_DIR / "images",
    PIPELINE_DIR / "dataset_full_data" / "train" / "images",
    PIPELINE_DIR / "dataset_full_data" / "val" / "images",
    PIPELINE_DIR / "dataset_full_data" / "test" / "images",
    PIPELINE_DIR / "dataset_sample_1000" / "train" / "images",
    PIPELINE_DIR / "dataset_sample_1000" / "val" / "images",
    PIPELINE_DIR / "dataset_sample_1000" / "test" / "images",
    REPO_ROOT / "images",
    REPO_ROOT / "coco_files" / "images",
    REPO_ROOT / "coco_files",
]

local_file_index: Dict[str, Path] = {}
for search_dir in CANDIDATE_SEARCH_DIRS:
    if search_dir.exists() and search_dir.is_dir():
        for p in search_dir.iterdir():
            if p.is_file() and p.stat().st_size > 0:
                if p.name not in local_file_index:
                    local_file_index[p.name] = p

logger.info(f"Local Image Indexer: Discovered {len(local_file_index)} unique cached image files across project directories.")

def resolve_or_download_image(img_record: dict, dest_dir: Path) -> Tuple[int, Path, Optional[str]]:
    """Resolves image from local cache. If already downloaded, skips downloading completely!"""
    img_id = img_record["id"]
    file_name = img_record.get("file_name") or f"image_{img_id}.jpg"
    dest_path = dest_dir / file_name
    
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return img_id, dest_path, None
    
    if file_name in local_file_index:
        src_path = local_file_index[file_name]
        try:
            if not dest_path.exists():
                try: os.symlink(os.path.abspath(src_path), dest_path)
                except OSError: shutil.copy2(src_path, dest_path)
            return img_id, dest_path, None
        except Exception:
            return img_id, src_path, None
            
    url = img_record.get("original_url") or img_record.get("url") or img_record.get("coco_url")
    if not url:
        return img_id, dest_path, "No URL provided and file does not exist locally"
    
    temp_path = dest_path.with_suffix(dest_path.suffix + ".tmp")
    try:
        resp = requests.get(url, timeout=30, verify=False)
        if resp.status_code == 200:
            with open(temp_path, "wb") as f: f.write(resp.content)
            os.replace(temp_path, dest_path)
            local_file_index[file_name] = dest_path
            return img_id, dest_path, None
        else:
            if temp_path.exists(): temp_path.unlink()
            return img_id, dest_path, f"HTTP Error {resp.status_code}"
    except Exception as e:
        if temp_path.exists(): temp_path.unlink()
        return img_id, dest_path, str(e)

# Audit: Separate cached vs truly missing images
already_cached_imgs = []
genuinely_missing_imgs = []

for img in raw_images:
    fname = img.get("file_name") or f"image_{img['id']}.jpg"
    if (TEST_IMAGES_DIR / fname).exists() and (TEST_IMAGES_DIR / fname).stat().st_size > 0:
        already_cached_imgs.append(img)
    elif fname in local_file_index:
        src = local_file_index[fname]
        dst = TEST_IMAGES_DIR / fname
        if not dst.exists():
            try: os.symlink(os.path.abspath(src), dst)
            except OSError: shutil.copy2(src, dst)
        already_cached_imgs.append(img)
    else:
        genuinely_missing_imgs.append(img)

logger.info(f"[CACHE AUDIT] Images Already Downloaded/Cached: {len(already_cached_imgs):,} / {len(raw_images):,} -> SKIPPED!")
logger.info(f"[CACHE AUDIT] Images Missing and Needing Download:  {len(genuinely_missing_imgs):,}")

if genuinely_missing_imgs:
    logger.info(f"Downloading {len(genuinely_missing_imgs)} missing images ({NUM_API_WORKERS} workers)...")
    with ThreadPoolExecutor(max_workers=NUM_API_WORKERS) as executor:
        futures = {executor.submit(resolve_or_download_image, img, TEST_IMAGES_DIR): img for img in genuinely_missing_imgs}
        pbar = tqdm(as_completed(futures), total=len(futures), desc="Downloading Missing Images", unit="img")
        for fut in pbar: fut.result()
    logger.info("Image downloads completed.")
else:
    logger.info("[SKIP] All test images are already downloaded and cached locally. 0 network calls made!")

# -------------------------------------------------------------------------
# Build Structured Test DataFrames
# -------------------------------------------------------------------------
test_images_records = []
test_annotations_records = []

for img in raw_images:
    img_id = img["id"]
    file_name = img.get("file_name") or f"image_{img_id}.jpg"
    img_path = TEST_IMAGES_DIR / file_name
    
    # Read actual image dimensions from disk for bulletproof scaling
    w, h = img.get("width"), img.get("height")
    if img_path.exists():
        try:
            with Image.open(img_path) as pil_im:
                actual_w, actual_h = pil_im.size
                if not w or not h:
                    w, h = actual_w, actual_h
        except Exception:
            actual_w, actual_h = w or 1920, h or 1080
    else:
        actual_w, actual_h = w or 1920, h or 1080
            
    anns = annotations_by_image_id.get(img_id, [])
    tag_counts_per_class = {name: 0 for name in class_names}
    
    for ann in anns:
        cid = ann["category_id"]
        cname = cat_id_to_cname.get(cid, "location_tag")
        c_idx = NEW_MODEL_CLASSES.index(cname) if cname in NEW_MODEL_CLASSES else 2
        tag_counts_per_class[cname] += 1
        is_white = (c_idx == white_tag_idx)
            
        bbox = ann.get("bbox", [0, 0, 0, 0])
        # Detect if bbox is normalized or pixel
        is_norm = (max(bbox) <= 1.05 and bbox[2] < 1.0)
        if is_norm:
            x1 = bbox[0] * actual_w
            y1 = bbox[1] * actual_h
            w_px = bbox[2] * actual_w
            h_px = bbox[3] * actual_h
        else:
            x1 = bbox[0]
            y1 = bbox[1]
            w_px = bbox[2]
            h_px = bbox[3]
            
        x2 = x1 + w_px
        y2 = y1 + h_px
        
        test_annotations_records.append({
            "annotation_id": ann.get("id"),
            "image_id": img_id,
            "file_name": file_name,
            "image_path": str(img_path),
            "category_id": cid,
            "class_idx": c_idx,
            "category_name": cname,
            "is_white_tag": is_white,
            "x1": round(x1, 2),
            "y1": round(y1, 2),
            "x2": round(x2, 2),
            "y2": round(y2, 2),
            "width": round(w_px, 2),
            "height": round(h_px, 2),
            "area": round(ann.get("area", w_px * h_px), 2),
        })
        
    img_entry = {
        "image_id": img_id,
        "file_name": file_name,
        "image_path": str(img_path),
        "exists_locally": img_path.exists(),
        "width": actual_w,
        "height": actual_h,
        "total_gt_tags": len(anns),
    }
    for cname in class_names:
        img_entry[f"{cname}_count"] = tag_counts_per_class[cname]
    test_images_records.append(img_entry)

df_test_images = pd.DataFrame(test_images_records)
df_test_annotations = pd.DataFrame(test_annotations_records)

test_img_csv = EVAL_OUTPUT_DIR / "test_images_manifest.csv"
test_ann_csv = EVAL_OUTPUT_DIR / "test_annotations_manifest.csv"
df_test_images.to_csv(test_img_csv, index=False)
df_test_annotations.to_csv(test_ann_csv, index=False)

logger.info(f"Test Manifests saved to: {test_img_csv} and {test_ann_csv}")

# Ensure test_samples list is globally available for all subsequent cells
test_samples = df_test_images[df_test_images["exists_locally"]].to_dict(orient="records")
logger.info(f"Globally registered {len(test_samples)} valid test image samples for evaluation.")


In [ ]:
# CELL 4: Comprehensive Test Dataset Annotation Counts & Audit
# "Always print the counts of annotations present and for evaluations also check if we correctly evaluating them"
logger.info("=" * 80)
logger.info("[CELL 4] Test Dataset Annotation Counts & Quality Verification")
logger.info("=" * 80)

total_test_images = len(df_test_images)
local_images_found = df_test_images["exists_locally"].sum()
total_gt_boxes = len(df_test_annotations)

cat_counts = df_test_annotations["category_name"].value_counts().to_dict()
white_tags_total = df_test_annotations["is_white_tag"].sum()
non_white_tags_total = total_gt_boxes - white_tags_total

print("\n" + "=" * 80)
print("TEST DATASET GROUND TRUTH AUDIT SUMMARY:")
print("=" * 80)
print(f"Total Test Images in Split:         {total_test_images:,}")
print(f"Test Images Verified Locally:       {local_images_found:,} / {total_test_images:,} ({local_images_found/max(1, total_test_images)*100:.1f}%)")
print(f"Total Ground Truth Bounding Boxes:  {total_gt_boxes:,}")
print("-" * 80)
print("GROUND TRUTH BREAKDOWN BY CATEGORY:")
print("-" * 80)

audit_rows = []
for cat in TARGET_CATEGORIES:
    count = cat_counts.get(cat, 0)
    # Check case variations
    if count == 0:
        for k, v in cat_counts.items():
            if k.lower() == cat.lower():
                count = v
                break
    pct = (count / max(1, total_gt_boxes)) * 100
    is_white = "Yes (White Tag - Evaluated by BOTH Models)" if cat == WHITE_TAG_CATEGORY else "No (Evaluated by New RF-DETR Model Only)"
    audit_rows.append({
        "Category Name": cat,
        "GT Box Count": count,
        "Percentage of Dataset": f"{pct:.1f}%",
        "Targeted Model Scope": is_white
    })

df_cat_audit = pd.DataFrame(audit_rows)
print(df_cat_audit.to_string(index=False))
print("-" * 80)
print(f"White Tags ('{WHITE_TAG_CATEGORY}'):  {white_tags_total:,} boxes ({white_tags_total/max(1, total_gt_boxes)*100:.1f}%)")
print(f"Multi-Class Tags (Blue Aisle + Blue Bay): {non_white_tags_total:,} boxes ({non_white_tags_total/max(1, total_gt_boxes)*100:.1f}%)")
print("=" * 80 + "\n")

# Distribution of boxes per image
boxes_per_image = df_test_images["total_gt_tags"]
print("Ground Truth Boxes per Image Distribution:")
print(f" - Min:    {boxes_per_image.min()}")
print(f" - Median: {boxes_per_image.median():.0f}")
print(f" - Mean:   {boxes_per_image.mean():.2f}")
print(f" - Max:    {boxes_per_image.max()}")
print(f" - Images with zero tags: {(boxes_per_image == 0).sum()}")
print("=" * 80)


In [ ]:
# CELL 5: Load Fine-Tuned Multi-Class RF-DETR Model (Extract LWDETR nn.Module)
# [FIX] Roboflow RFDETRBase wrapper.model is rfdetr_main.Model, while the underlying PyTorch nn.Module is wrapper.model.model (LWDETR)
logger.info("=" * 80)
logger.info(f"[CELL 5] Loading RF-DETR Multi-Class Model from Local Path: {RFDETR_CHECKPOINT_PATH}")
logger.info("=" * 80)

rfdetr_model = None
num_classes = len(TARGET_CATEGORIES)

# Clean up corrupted zero-byte cache files if any exist
for bad_pth in ["rf-detr-base.pth", "rf-detr-base-coco.pth", os.path.expanduser("~/.roboflow/models/rf-detr-base.pth")]:
    if os.path.exists(bad_pth) and os.path.getsize(bad_pth) < 1024 * 1024:
        try: os.remove(bad_pth)
        except Exception: pass

if RFDETR_CHECKPOINT_PATH and RFDETR_CHECKPOINT_PATH.exists():
    ckpt_size = RFDETR_CHECKPOINT_PATH.stat().st_size
    logger.info(f"[SKIP DOWNLOAD] Local fine-tuned checkpoint found ({ckpt_size / (1024**2):.1f} MB). Skipping all online weights downloads.")
    
    try:
        import torch.nn as nn
        from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
        from rfdetr import main as rfdetr_main
        from rfdetr.models.lwdetr import LWDETR
        
        # Hard-block background downloads of default COCO weights from Roboflow / GitHub
        if hasattr(RFDETRBase, "maybe_download_pretrain_weights"):
            RFDETRBase.maybe_download_pretrain_weights = lambda self: None
        if hasattr(RFDETRBase, "load_pretrain_weights"):
            RFDETRBase.load_pretrain_weights = lambda self: None
            
        # 1. Patch reinitialize_detection_head (matching multi_class_train_rfdetr.ipynb Cell 14)
        def _fixed_reinitialize(self, n_classes):
            lw = self.model  # LWDETR nn.Module inside Model wrapper
            dev = next(lw.parameters()).device if list(lw.parameters()) else torch.device("cpu")
            if hasattr(lw, "class_embed"):
                in_feat = lw.class_embed.in_features
                lw.class_embed = nn.Linear(in_feat, n_classes).to(dev)
                nn.init.normal_(lw.class_embed.weight, std=0.01)
                nn.init.zeros_(lw.class_embed.bias)
            if hasattr(lw, "transformer") and hasattr(lw.transformer, "enc_out_class_embed"):
                enc = lw.transformer.enc_out_class_embed
                if isinstance(enc, nn.ModuleList):
                    for i in range(len(enc)):
                        in_feat = enc[i].in_features
                        enc[i] = nn.Linear(in_feat, n_classes).to(dev)
                        nn.init.normal_(enc[i].weight, std=0.01)
                        nn.init.zeros_(enc[i].bias)
                elif isinstance(enc, nn.Linear):
                    in_feat = enc.in_features
                    lw.transformer.enc_out_class_embed = nn.Linear(in_feat, n_classes).to(dev)
                    nn.init.normal_(lw.transformer.enc_out_class_embed.weight, std=0.01)
                    nn.init.zeros_(lw.transformer.enc_out_class_embed.bias)
                    
        rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
        
        # 2. Patch LWDETR.load_state_dict for clean shape matching
        def _safe_lwdetr_load_state_dict(self, state_dict, strict=True):
            model_state = self.state_dict()
            filtered = {}
            for k, v in state_dict.items():
                clean_k = k
                if clean_k.startswith("module."): clean_k = clean_k[7:]
                if clean_k.startswith("model.model."): clean_k = clean_k[12:]
                elif clean_k.startswith("model."): clean_k = clean_k[6:]
                
                if clean_k in model_state and model_state[clean_k].shape == v.shape:
                    filtered[clean_k] = v
                elif k in model_state and model_state[k].shape == v.shape:
                    filtered[k] = v
            return torch.nn.Module.load_state_dict(self, filtered, strict=False)
            
        LWDETR.load_state_dict = _safe_lwdetr_load_state_dict
        
        # 3. Model Architecture Instantiation
        model_cls_map = {
            "nano": RFDETRNano,
            "small": RFDETRSmall,
            "medium": RFDETRMedium,
            "base": RFDETRBase,
            "large": RFDETRLarge
        }
        ModelClass = model_cls_map.get(MODEL_SIZE.lower(), RFDETRBase)
        logger.info(f"Initializing RF-DETR {MODEL_SIZE.capitalize()} architecture (num_classes={num_classes}, resolution={RESOLUTION})...")
        
        wrapper = ModelClass(num_classes=num_classes, resolution=RESOLUTION, pretrain_weights=None)
        
        # Ensure detection heads match num_classes
        if hasattr(wrapper, "model") and hasattr(wrapper.model, "reinitialize_detection_head"):
            wrapper.model.reinitialize_detection_head(num_classes)
            
        # 4. Extract the underlying PyTorch nn.Module (LWDETR)
        # Note: wrapper is RFDETRBase, wrapper.model is rfdetr_main.Model, wrapper.model.model is LWDETR (nn.Module)
        if hasattr(wrapper, "model") and hasattr(wrapper.model, "model") and isinstance(wrapper.model.model, torch.nn.Module):
            lwdetr = wrapper.model.model
        elif hasattr(wrapper, "model") and isinstance(wrapper.model, torch.nn.Module):
            lwdetr = wrapper.model
        elif isinstance(wrapper, torch.nn.Module):
            lwdetr = wrapper
        else:
            raise AttributeError(f"Could not extract torch.nn.Module from {type(wrapper)}")
            
        # 5. Load Fine-Tuned Weights into LWDETR
        ckpt = torch.load(str(RFDETR_CHECKPOINT_PATH), map_location="cpu", weights_only=False)
        if isinstance(ckpt, dict):
            if "model_state_dict" in ckpt and isinstance(ckpt["model_state_dict"], dict):
                state = ckpt["model_state_dict"]
            elif "model" in ckpt and isinstance(ckpt["model"], dict):
                state = ckpt["model"]
            else:
                state = ckpt
        else:
            state = ckpt
            
        load_result = lwdetr.load_state_dict(state, strict=False)
        logger.info(f"Fine-tuned weights loaded into LWDETR: {load_result}")
        
        lwdetr.to(device)
        lwdetr.eval()
        rfdetr_model = lwdetr
        
        param_count = sum(p.numel() for p in rfdetr_model.parameters()) / 1e6
        logger.info(f"RF-DETR Multi-Class Model Ready on {device} ({param_count:.1f}M parameters).")
        
    except Exception as e:
        logger.error(f"Failed to load RF-DETR model: {e}")
        rfdetr_model = None
else:
    logger.warning(f"RF-DETR checkpoint not found at: {RFDETR_CHECKPOINT_PATH}")


In [ ]:
# CELL 6: Evaluate New RF-DETR Multi-Class Model on Held-Out Test Set (High-Throughput Batched Pipeline)
# Performance Optimization:
#   - Mini-batch inference (batch_size=4 or 8) saturates GPU Tensor Cores
#   - PyTorch DataLoader with asynchronous worker prefetching hides disk I/O and image resizing
#   - torch.inference_mode() + torch.backends.cudnn.benchmark = True for max FPS (6-12+ images/sec)
logger.info("=" * 80)
logger.info("[CELL 6] New Model Inference Evaluation: RF-DETR vs. COCO Ground Truth")
logger.info(f"Classes: {NEW_MODEL_CLASSES}")
logger.info(f"Evaluation Confidence Threshold: {CONFIDENCE_THRESHOLD} | IoU Threshold: {IOU_THRESHOLD}")
logger.info(f"Evaluation Batch Size: {EVAL_BATCH_SIZE} | DataLoader Workers: {NUM_EVAL_WORKERS}")
logger.info("=" * 80)

# Enable cuDNN autotuner for static input shape (1008x1008)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    """Converts cx, cy, w, h to x1, y1, x2, y2 coordinates."""
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)

def compute_box_iou(b1: List[float], b2: List[float]) -> float:
    """Computes Intersection-over-Union (IoU) between two bounding boxes [x1, y1, x2, y2]."""
    xA, yA = max(b1[0], b2[0]), max(b1[1], b2[1])
    xB, yB = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0.0, xB - xA) * max(0.0, yB - yA)
    area1 = max(0.0, b1[2] - b1[0]) * max(0.0, b1[3] - b1[1])
    area2 = max(0.0, b2[2] - b2[0]) * max(0.0, b2[3] - b2[1])
    union = area1 + area2 - inter
    return (inter / union) if union > 0 else 0.0

# -------------------------------------------------------------------------
# Single-Image Prediction Function for RF-DETR Multi-Class Model
# -------------------------------------------------------------------------
def predict_tags_rfdetr(
    img_path: str,
    model: Optional[torch.nn.Module] = None,
    conf_thresh: float = CONFIDENCE_THRESHOLD,
    resolution: int = RESOLUTION
) -> List[Dict[str, Any]]:
    """
    Predicts multi-class tags on a single image using RF-DETR with parameterized confidence threshold.
    
    Classes:
        0: blue_aisle
        1: blue_bay
        2: location_tag
    """
    if model is None:
        model = rfdetr_model
    if model is None or not os.path.exists(img_path):
        return []
        
    pil_img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = pil_img.size
    
    resized = pil_img.resize((resolution, resolution), Image.BILINEAR)
    x = TF.to_tensor(resized).unsqueeze(0)
    x = TF.normalize(x, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).to(device)
    
    with torch.inference_mode():
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            out = model(x)
            
    logits = out["pred_logits"][0]
    boxes = out["pred_boxes"][0]
    
    scores, classes = logits.sigmoid().max(-1)
    keep = scores > conf_thresh
    
    preds = []
    for s, c, (cx, cy, w, h) in zip(scores[keep], classes[keep], boxes[keep]):
        x1 = float(max(0, (cx - w / 2) * orig_w))
        y1 = float(max(0, (cy - h / 2) * orig_h))
        x2 = float(min(orig_w, (cx + w / 2) * orig_w))
        y2 = float(min(orig_h, (cy + h / 2) * orig_h))
        cid = int(c)
        tag_name = NEW_MODEL_CLASSES[cid] if cid < len(NEW_MODEL_CLASSES) else f"class_{cid}"
        preds.append({
            "tag": tag_name,
            "class_id": cid,
            "score": round(float(s), 4),
            "box": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]
        })
        
    if len(preds) == 0 and len(scores) > 0 and conf_thresh <= 0.15:
        max_sc = scores.max().item()
        if max_sc > 0.03:
            top_k = scores.topk(min(3, len(scores)))
            for s, idx in zip(top_k.values, top_k.indices):
                if s > 0.05:
                    cx, cy, w, h = boxes[idx]
                    cid = int(classes[idx])
                    tag_name = NEW_MODEL_CLASSES[cid] if cid < len(NEW_MODEL_CLASSES) else f"class_{cid}"
                    preds.append({
                        "tag": tag_name,
                        "class_id": cid,
                        "score": round(float(s), 4),
                        "box": [
                            round(float(max(0, (cx - w / 2) * orig_w)), 1),
                            round(float(max(0, (cy - h / 2) * orig_h)), 1),
                            round(float(min(orig_w, (cx + w / 2) * orig_w)), 1),
                            round(float(min(orig_h, (cy + h / 2) * orig_h)), 1)
                        ]
                    })
    return preds

# -------------------------------------------------------------------------
# High-Speed Prefetching Dataset for Fast Batched Evaluation
# -------------------------------------------------------------------------
class RFDETRTestDataset(torch.utils.data.Dataset):
    """Prefetches and preprocesses test images in background CPU threads for max GPU saturation."""
    def __init__(self, samples: List[dict], resolution: int = 1008):
        self.samples = samples
        self.resolution = resolution
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_path = sample["image_path"]
        
        # High-speed OpenCV decoding with PIL fallback
        cv_img = cv2.imread(img_path)
        if cv_img is not None:
            actual_h, actual_w = cv_img.shape[:2]
            resized = cv2.resize(cv_img, (self.resolution, self.resolution), interpolation=cv2.INTER_LINEAR)
            rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
            tensor = torch.from_numpy(rgb).permute(2, 0, 1).float().div_(255.0)
        else:
            with Image.open(img_path).convert("RGB") as pil_im:
                actual_w, actual_h = pil_im.size
                resized = pil_im.resize((self.resolution, self.resolution), Image.BILINEAR)
                tensor = TF.to_tensor(resized)
                
        tensor = TF.normalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        return tensor, actual_w, actual_h, idx

# -------------------------------------------------------------------------
# Systematic High-Throughput Evaluation Loop (Batched Inference)
# -------------------------------------------------------------------------
def evaluate_rfdetr(
    conf_thresh: float = CONFIDENCE_THRESHOLD,
    iou_thresh: float = IOU_THRESHOLD,
    batch_size: int = EVAL_BATCH_SIZE,
    num_workers: int = NUM_EVAL_WORKERS,
    max_samples: Optional[int] = None
) -> Tuple[List[dict], Dict[str, dict], pd.DataFrame]:
    """
    High-throughput evaluation of RF-DETR against Ground Truth using mini-batches and prefetching.
    Yields 6-12+ images/sec on modern GPUs.
    """
    class_stats = {
        cname: {"gt": 0, "preds": 0, "tp": 0, "fp": 0, "fn": 0}
        for cname in NEW_MODEL_CLASSES
    }
    
    total_loc_gt = 0
    total_loc_preds = 0
    total_loc_tp = 0
    total_loc_fp = 0
    class_confusion_count = 0
    
    eval_results = []
    test_samples = df_test_images[df_test_images["exists_locally"]].to_dict(orient="records")
    if max_samples:
        test_samples = test_samples[:max_samples]
        
    logger.info(f"Starting High-Speed RF-DETR Evaluation on {len(test_samples)} images...")
    logger.info(f"Inference Mode: Batched (Batch Size = {batch_size}, Prefetch Workers = {num_workers}, Resolution = {RESOLUTION}x{RESOLUTION})")
    
    # Pre-register Ground Truth stats
    for sample in test_samples:
        img_id = sample["image_id"]
        gt_records = df_test_annotations[df_test_annotations["image_id"] == img_id].to_dict(orient="records")
        total_loc_gt += len(gt_records)
        for r in gt_records:
            c = r["category_name"]
            if c in class_stats:
                class_stats[c]["gt"] += 1

    # Initialize PyTorch DataLoader for asynchronous multi-worker prefetching
    eval_dataset = RFDETRTestDataset(test_samples, resolution=RESOLUTION)
    eval_loader = torch.utils.data.DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=False
    )
    
    t0 = time.time()
    
    with torch.inference_mode():
        for batch_tensors, orig_ws, orig_hs, sample_indices in tqdm(
            eval_loader,
            desc=f"RF-DETR Fast Eval (batch={batch_size})",
            unit="img",
            unit_scale=batch_size
        ):
            batch_tensors = batch_tensors.to(device, non_blocking=True)
            
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                out = rfdetr_model(batch_tensors)
                
            batch_logits = out["pred_logits"]   # [B, queries, num_classes]
            batch_boxes = out["pred_boxes"]     # [B, queries, 4]
            
            # Process each image in the mini-batch
            for b_idx in range(len(sample_indices)):
                s_idx = sample_indices[b_idx].item()
                sample = test_samples[s_idx]
                img_id = sample["image_id"]
                file_name = sample["file_name"]
                orig_w = float(orig_ws[b_idx].item())
                orig_h = float(orig_hs[b_idx].item())
                
                gt_records = df_test_annotations[df_test_annotations["image_id"] == img_id].to_dict(orient="records")
                gt_boxes = [[r["x1"], r["y1"], r["x2"], r["y2"]] for r in gt_records]
                gt_classes = [r["category_name"] for r in gt_records]
                gt_class_indices = [r["class_idx"] for r in gt_records]
                
                logits = batch_logits[b_idx]
                boxes_n = batch_boxes[b_idx]
                
                scores, classes = logits.sigmoid().max(-1)
                keep = scores > conf_thresh
                
                preds = []
                for s, c, (cx, cy, w, h) in zip(scores[keep], classes[keep], boxes_n[keep]):
                    x1 = float(max(0, (cx - w / 2) * orig_w))
                    y1 = float(max(0, (cy - h / 2) * orig_h))
                    x2 = float(min(orig_w, (cx + w / 2) * orig_w))
                    y2 = float(min(orig_h, (cy + h / 2) * orig_h))
                    cid = int(c)
                    tag_name = NEW_MODEL_CLASSES[cid] if cid < len(NEW_MODEL_CLASSES) else f"class_{cid}"
                    preds.append({
                        "tag": tag_name,
                        "class_id": cid,
                        "score": round(float(s), 4),
                        "box": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]
                    })
                    
                # Bipartite Matching against Ground Truth
                matched_gt_indices = set()
                matched_loc_gt = set()
                annotated_preds = []
                
                sorted_preds = sorted(preds, key=lambda p: p["score"], reverse=True)
                total_loc_preds += len(sorted_preds)
                
                for p in sorted_preds:
                    pb = p["box"]
                    psc = p["score"]
                    pcl = p["tag"]
                    pcid = p["class_id"]
                    
                    if pcl in class_stats:
                        class_stats[pcl]["preds"] += 1
                        
                    # Class-Aware Bipartite Matching
                    best_iou = 0.0
                    best_gt_idx = -1
                    for g_i, (gb, gc, gcid) in enumerate(zip(gt_boxes, gt_classes, gt_class_indices)):
                        if g_i in matched_gt_indices:
                            continue
                        if gcid == pcid or gc.lower().replace(" ", "_") == pcl.lower().replace(" ", "_"):
                            iou = compute_box_iou(pb, gb)
                            if iou > best_iou:
                                best_iou = iou
                                best_gt_idx = g_i
                                
                    is_correct = (best_iou >= iou_thresh and best_gt_idx >= 0)
                    if is_correct:
                        class_stats[pcl]["tp"] += 1
                        matched_gt_indices.add(best_gt_idx)
                    else:
                        class_stats[pcl]["fp"] += 1
                        
                    # Class confusion check
                    confusion_with = None
                    if not is_correct:
                        for g_i, (gb, gc, gcid) in enumerate(zip(gt_boxes, gt_classes, gt_class_indices)):
                            iou = compute_box_iou(pb, gb)
                            if iou >= iou_thresh:
                                confusion_with = gc
                                class_confusion_count += 1
                                break
                                
                    # Localization check (any class)
                    best_any_iou = 0.0
                    best_any_idx = -1
                    for g_i, gb in enumerate(gt_boxes):
                        if g_i in matched_loc_gt: continue
                        iou = compute_box_iou(pb, gb)
                        if iou > best_any_iou:
                            best_any_iou = iou
                            best_any_idx = g_i
                    if best_any_iou >= iou_thresh and best_any_idx >= 0:
                        total_loc_tp += 1
                        matched_loc_gt.add(best_any_idx)
                    else:
                        total_loc_fp += 1
                        
                    annotated_preds.append({
                        "box": pb,
                        "score": psc,
                        "class_name": pcl,
                        "is_correct": is_correct,
                        "matched_gt_class": gt_classes[best_gt_idx] if best_gt_idx >= 0 else confusion_with,
                        "iou": round(best_iou if is_correct else best_any_iou, 3)
                    })
                    
                    rfdetr_predictions_list.append({
                        "image_id": img_id,
                        "file_name": file_name,
                        "model": "RF-DETR (New)",
                        "predicted_class": pcl,
                        "confidence": psc,
                        "x1": round(pb[0], 2),
                        "y1": round(pb[1], 2),
                        "x2": round(pb[2], 2),
                        "y2": round(pb[3], 2),
                        "is_correct": is_correct,
                        "iou": round(best_iou, 3)
                    })
                    
                # Unmatched GT boxes = False Negatives
                for g_i, (gb, gc) in enumerate(zip(gt_boxes, gt_classes)):
                    if g_i not in matched_gt_indices:
                        if gc in class_stats:
                            class_stats[gc]["fn"] += 1
                            
                eval_results.append({
                    "image_id": img_id,
                    "file_name": file_name,
                    "gt_boxes": gt_boxes,
                    "gt_classes": gt_classes,
                    "annotated_preds": annotated_preds
                })
                
    duration = time.time() - t0
    fps = len(test_samples) / max(duration, 0.001)
    
    total_gt = sum(st["gt"] for st in class_stats.values())
    total_preds = sum(st["preds"] for st in class_stats.values())
    total_tp = sum(st["tp"] for st in class_stats.values())
    total_fp = sum(st["fp"] for st in class_stats.values())
    total_fn = sum(st["fn"] for st in class_stats.values())
    
    overall_p = (total_tp / max(1, total_preds)) * 100
    overall_r = (total_tp / max(1, total_gt)) * 100
    overall_f1 = (2 * overall_p * overall_r) / max(1e-5, overall_p + overall_r)
    loc_r = (total_loc_tp / max(1, total_loc_gt)) * 100
    loc_p = (total_loc_tp / max(1, total_loc_preds)) * 100
    
    print("\n" + "=" * 85)
    print(f"NEW MODEL (RF-DETR) EVALUATION RESULTS (Confidence Threshold: {conf_thresh}):")
    print("=" * 85)
    print(f"Total Ground Truth Target Boxes:   {total_gt:,}")
    print(f"Total Predictions Generated:       {total_preds:,}")
    print(f"True Positives (Correct Boxes):    {total_tp:,}")
    print(f"False Positives (Incorrect/Noise): {total_fp:,}")
    print(f"False Negatives (Missed Boxes):    {total_fn:,}")
    print(f"Overall Multi-Class Precision:     {overall_p:.2f}%")
    print(f"Overall Multi-Class Recall:        {overall_r:.2f}%")
    print(f"Overall Multi-Class F1 Score:      {overall_f1:.2f}%")
    print(f"Class-Agnostic Tag Localization:   {total_loc_tp:,} / {total_loc_gt:,} ({loc_r:.2f}% Recall, {loc_p:.2f}% Precision)")
    print(f"Cross-Category Label Confusion:    {class_confusion_count} instances")
    print(f"Inference Throughput:              {fps:.1f} FPS ({duration:.1f}s for {len(test_samples)} images)")
    print("-" * 85)
    print("PER-CATEGORY BREAKDOWN (RF-DETR):")
    print("-" * 85)
    
    rows = []
    for cname in NEW_MODEL_CLASSES:
        st = class_stats[cname]
        cp = (st["tp"] / max(1, st["preds"])) * 100
        cr = (st["tp"] / max(1, st["gt"])) * 100
        cf1 = (2 * cp * cr) / max(1e-5, cp + cr)
        rows.append({
            "Category": cname,
            "Ground Truth": st["gt"],
            "Predicted": st["preds"],
            "Correct (TP)": st["tp"],
            "False Alerts (FP)": st["fp"],
            "Missed (FN)": st["fn"],
            "Precision": f"{cp:.1f}%",
            "Recall": f"{cr:.1f}%",
            "F1 Score": f"{cf1:.1f}%"
        })
    df_cat = pd.DataFrame(rows)
    print(df_cat.to_string(index=False))
    print("=" * 85)
    
    return eval_results, class_stats, df_cat

# Containers for predictions
rfdetr_predictions_list = []

# Execute high-throughput evaluation using configured parameters
rfdetr_eval_results, rfdetr_class_stats, df_rf_cat = evaluate_rfdetr(
    conf_thresh=CONFIDENCE_THRESHOLD,
    iou_thresh=IOU_THRESHOLD,
    batch_size=EVAL_BATCH_SIZE,
    num_workers=NUM_EVAL_WORKERS
)


In [ ]:
# CELL 7: Old Model REST API Endpoint Inference (Strictly Using Test COCO Annotations)
# [CONFIG] "for old model inference use test coco files annotations"
# [RULE] If an image is already in location_tag_detections.csv, SKIP IT completely!
logger.info("=" * 80)
logger.info("[CELL 7] Old Model Inference: Production REST Endpoint Calling for Test COCO Images")
logger.info("=" * 80)

# =========================================================================
# 1. CONFIG: TEST COCO Annotations & Endpoint Settings
# =========================================================================
API_URL = "https://prod-itemrecognitionservice.cld.samsclub.com/v3/location_tag"
CLUB_ID = "4822"
API_TIMEOUT = 60
NUM_API_WORKERS = 16
API_OUTPUT_CSV = REPO_ROOT / "location_tag_detections.csv"

# Test Images Folder and COCO annotations file
TEST_COCO_JSON = TEST_ANN_PATH
TEST_IMAGE_FOLDER = TEST_IMAGES_DIR

logger.info(f"Target REST Endpoint:       {API_URL}")
logger.info(f"Test COCO Annotations File: {TEST_COCO_JSON}")
logger.info(f"Test Images Folder:         {TEST_IMAGE_FOLDER}")
logger.info(f"Parallel Worker Threads:    {NUM_API_WORKERS} | Club ID: {CLUB_ID}")

# =========================================================================
# 2. EXTRACT IMAGES STRICTLY FROM TEST COCO ANNOTATIONS
# =========================================================================
with open(TEST_COCO_JSON, "r", encoding="utf-8") as f:
    test_coco_manifest = json.load(f)

test_coco_image_list = test_coco_manifest.get("images", [])
logger.info(f"Loaded {len(test_coco_image_list)} image entries strictly from TEST COCO annotations ({TEST_COCO_JSON.name})")

# Resolve local file path for each test image entry from COCO annotations
test_image_items = []
missing_local_files = []

for img_meta in test_coco_image_list:
    img_id = img_meta["id"]
    file_name = img_meta.get("file_name") or f"image_{img_id}.jpg"
    
    # 1. Check primary test images folder
    target_path = TEST_IMAGE_FOLDER / file_name
    if not (target_path.exists() and target_path.stat().st_size > 0):
        # 2. Check local file index cache
        if file_name in local_file_index:
            target_path = local_file_index[file_name]
            
    if target_path.exists() and target_path.stat().st_size > 0:
        test_image_items.append({
            "image_id": img_id,
            "file_name": file_name,
            "image_path": Path(target_path)
        })
    else:
        missing_local_files.append(file_name)

logger.info(f"Test COCO Images Resolved Locally: {len(test_image_items):,} / {len(test_coco_image_list):,}")
if missing_local_files:
    logger.warning(f"{len(missing_local_files)} test images from COCO annotations were not found locally on disk.")

# Optional Sampling (set SAMPLE_SIZE in Cell 2 if desired; None = process all test images)
if SAMPLE_SIZE is not None and len(test_image_items) > SAMPLE_SIZE:
    random.seed(42)
    selected_test_items = random.sample(test_image_items, SAMPLE_SIZE)
    logger.info(f"Sample size applied: Processing {len(selected_test_items)} test images from COCO annotations.")
else:
    selected_test_items = test_image_items
    logger.info(f"Processing ALL {len(selected_test_items)} test images from COCO annotations.")

# =========================================================================
# 3. ENDPOINT CLIENT HELPERS
# =========================================================================
def image_to_base64(image_path: Path) -> str:
    """Encodes local image to base64 UTF-8 string."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def call_location_tag_api(image_path: Path, club_id: str = CLUB_ID, timeout: int = API_TIMEOUT) -> dict:
    """Dispatches REST API request to production location_tag endpoint."""
    payload = {
        "session_id": str(uuid.uuid4()),
        "club_id": club_id,
        "requests": [{
            "image": {"content": image_to_base64(image_path)},
            "features": [{"type": "LOCATION_TAG", "maxResults": 50}]
        }]
    }
    response = requests.post(
        API_URL,
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=timeout,
        verify=False
    )
    response.raise_for_status()
    return response.json()

def process_test_coco_image(item: Dict[str, Any], club_id: str = CLUB_ID) -> List[Dict[str, Any]]:
    """Calls endpoint for one image entry from the test COCO annotations and returns formatted detection rows."""
    image_path = item["image_path"]
    file_name = item["file_name"]
    image_id = item["image_id"]
    
    try:
        response = call_location_tag_api(image_path=image_path, club_id=club_id)
        rows = []
        responses = response.get("responses", [])
        
        if not responses:
            return [{
                "image_id": image_id,
                "image_path": str(image_path),
                "file_name": file_name,
                "tag_detected": False,
                "tag_recognized": False,
                "detection_id": None,
                "text": None,
                "score": None,
                "x1": None,
                "y1": None,
                "x2": None,
                "y2": None,
                "num_detections": 0,
                "error": None
            }]
            
        for api_result in responses:
            tag_detected = api_result.get("tag_detected", False)
            tag_recognized = api_result.get("tag_recognized", False)
            results = api_result.get("results", [])
            num_detections = len(results)
            
            if num_detections == 0:
                rows.append({
                    "image_id": image_id,
                    "image_path": str(image_path),
                    "file_name": file_name,
                    "tag_detected": tag_detected,
                    "tag_recognized": tag_recognized,
                    "detection_id": None,
                    "text": None,
                    "score": None,
                    "x1": None,
                    "y1": None,
                    "x2": None,
                    "y2": None,
                    "num_detections": 0,
                    "error": None
                })
            else:
                for idx, detection in enumerate(results):
                    rows.append({
                        "image_id": image_id,
                        "image_path": str(image_path),
                        "file_name": file_name,
                        "tag_detected": tag_detected,
                        "tag_recognized": tag_recognized,
                        "detection_id": idx,
                        "text": detection.get("text"),
                        "score": detection.get("score"),
                        "x1": detection.get("x1"),
                        "y1": detection.get("y1"),
                        "x2": detection.get("x2"),
                        "y2": detection.get("y2"),
                        "num_detections": num_detections,
                        "error": None
                    })
        return rows
    except Exception as e:
        return [{
            "image_id": image_id,
            "image_path": str(image_path),
            "file_name": file_name,
            "tag_detected": None,
            "tag_recognized": None,
            "detection_id": None,
            "text": None,
            "score": None,
            "x1": None,
            "y1": None,
            "x2": None,
            "y2": None,
            "num_detections": None,
            "error": str(e)
        }]

# =========================================================================
# 4. SMART INCREMENTAL SKIP: Skip images already in location_tag_detections.csv
# =========================================================================
existing_df = None
already_processed_files = set()

if API_OUTPUT_CSV.exists() and API_OUTPUT_CSV.stat().st_size > 100:
    try:
        existing_df = pd.read_csv(API_OUTPUT_CSV)
        if "file_name" in existing_df.columns:
            already_processed_files = set(existing_df["file_name"].dropna().unique())
            logger.info(f"Loaded existing detections from {API_OUTPUT_CSV}: {len(already_processed_files)} unique images already recorded.")
    except Exception as e:
        logger.warning(f"Could not read existing {API_OUTPUT_CSV}: {e}")

# Filter out already processed test COCO images
items_to_query = [item for item in selected_test_items if item["file_name"] not in already_processed_files]

logger.info("=" * 80)
logger.info(f"[ENDPOINT AUDIT - TEST COCO ANNOTATIONS]")
logger.info(f" - Total Test Images from COCO Annotations: {len(selected_test_items):,}")
logger.info(f" - Images Already Processed (in CSV):       {len(already_processed_files):,} -> SKIPPED!")
logger.info(f" - New Test Images to Query via API:        {len(items_to_query):,}")
logger.info("=" * 80)

# =========================================================================
# 5. EXECUTE ENDPOINT CALLS WITH THREADPOOL
# =========================================================================
new_rows = []
if items_to_query:
    logger.info(f"Querying endpoint for {len(items_to_query)} test COCO images ({NUM_API_WORKERS} workers)...")
    api_start_time = time.time()
    with ThreadPoolExecutor(max_workers=NUM_API_WORKERS) as executor:
        futures = {executor.submit(process_test_coco_image, item, CLUB_ID): item for item in items_to_query}
        pbar = tqdm(as_completed(futures), total=len(futures), desc="Old Model Inference (Test COCO)", unit="image")
        for future in pbar:
            try:
                res = future.result()
                new_rows.extend(res)
            except Exception as e:
                logger.warning(f"Worker exception: {e}")
    elapsed = time.time() - api_start_time
    logger.info(f"API calls completed in {elapsed:.2f}s ({(len(items_to_query)/max(0.1, elapsed)):.1f} img/sec).")
else:
    logger.info("[SKIP] All test COCO images already have detections in CSV. 0 API calls needed!")

# Merge newly queried rows with existing DataFrame
if new_rows:
    df_new = pd.DataFrame(new_rows)
    if existing_df is not None and not existing_df.empty:
        df_old_model_raw = pd.concat([existing_df, df_new], ignore_index=True)
    else:
        df_old_model_raw = df_new
    df_old_model_raw.to_csv(API_OUTPUT_CSV, index=False)
    logger.info(f"Updated and saved detections to: {API_OUTPUT_CSV} ({len(df_old_model_raw)} total rows)")
else:
    df_old_model_raw = existing_df if existing_df is not None else pd.DataFrame()

# =========================================================================
# 6. SUMMARY OF OLD MODEL PREDICTIONS ON TEST COCO DATASET
# =========================================================================
print("\n" + "=" * 80)
print("OLD MODEL INFERENCE SUMMARY (TEST COCO ANNOTATIONS DATASET):")
print("=" * 80)
print(f"Source Annotations File:          {TEST_COCO_JSON.name}")
print(f"Total Rows in Detections Table:   {len(df_old_model_raw):,}")
print(f"Unique Test Images Evaluated:     {df_old_model_raw['file_name'].nunique() if 'file_name' in df_old_model_raw.columns else 0:,}")

if "error" in df_old_model_raw.columns:
    errors_count = df_old_model_raw["error"].notna().sum()
    print(f"Failed API Calls (Errors/Timeouts): {errors_count}")

valid_boxes_df = df_old_model_raw[df_old_model_raw["x1"].notna()] if "x1" in df_old_model_raw.columns else pd.DataFrame()
print(f"Total Bounding Boxes Detected:    {len(valid_boxes_df):,}")
if "file_name" in df_old_model_raw.columns and not valid_boxes_df.empty:
    imgs_with_det = valid_boxes_df["file_name"].nunique()
    print(f"Images with at Least 1 Detection: {imgs_with_det:,} / {df_old_model_raw['file_name'].nunique():,}")
print("=" * 80)


In [ ]:
# CELL 8: Systematic Evaluation of Old Model against Test Ground Truth (Supporting Multiple Detections)
# "Old model inference can detect multiple detections"
# "Old model is built only on location_tag white tags only, now I want to compare the models"
# "for evaluations also check if we correctly evaluating them"
logger.info("=" * 80)
logger.info("[CELL 8] Old Model Inference Evaluation: REST Endpoint vs. COCO Ground Truth")
logger.info("=" * 80)

# Safety check: Guarantee test_samples is defined in globals
if "test_samples" not in globals() or not test_samples:
    test_samples = df_test_images[df_test_images["exists_locally"]].to_dict(orient="records")
logger.info(f"Evaluating Old Model on {len(test_samples)} test image samples...")

# =========================================================================
# 1. BUILD MULTIPLE DETECTIONS LOOKUP TABLE BY FILE NAME
# Multiple detections per image from endpoint are preserved and indexed
# =========================================================================
old_preds_by_file = defaultdict(list)
for _, row in df_old_model_raw[df_old_model_raw["x1"].notna()].iterrows():
    fname = row["file_name"]
    x1, y1, x2, y2 = float(row["x1"]), float(row["y1"]), float(row["x2"]), float(row["y2"])
    score = float(row["score"]) if pd.notna(row["score"]) else 0.70
    old_preds_by_file[fname].append({
        "box_raw": [x1, y1, x2, y2],
        "score": score,
        "text": row.get("text", "")
    })

# Audit Multiple Detections Capacity of Old Model
multi_det_distribution = [len(old_preds_by_file.get(s["file_name"], [])) for s in test_samples]
img_multi_2plus = sum(1 for c in multi_det_distribution if c >= 2)
img_multi_5plus = sum(1 for c in multi_det_distribution if c >= 5)
img_multi_10plus = sum(1 for c in multi_det_distribution if c >= 10)
max_dets_single_img = max(multi_det_distribution) if multi_det_distribution else 0
avg_dets_per_img = np.mean(multi_det_distribution) if multi_det_distribution else 0.0

logger.info("=" * 80)
logger.info("OLD MODEL MULTIPLE DETECTIONS AUDIT:")
logger.info(f" - Average Detections per Image:       {avg_dets_per_img:.2f} tags/img")
logger.info(f" - Maximum Detections in Single Image: {max_dets_single_img} tags")
logger.info(f" - Images with Multiple Detections (>=2): {img_multi_2plus:,} ({img_multi_2plus/max(1, len(test_samples))*100:.1f}%)")
logger.info(f" - Images with Dense Clusters (>=5):    {img_multi_5plus:,}")
logger.info(f" - Images with Heavy Clusters (>=10):   {img_multi_10plus:,}")
logger.info("=" * 80)

# Evaluation Counters for Old Model:
old_white_gt = 0
old_white_preds = 0
old_white_tp = 0
old_white_fp = 0
old_white_fn = 0

old_any_gt = 0
old_any_preds = 0
old_any_tp = 0
old_any_fp = 0
old_any_fn = 0

old_hit_blue_aisle = 0
old_hit_blue_bay = 0

old_model_eval_results = []
old_predictions_list = []  # For mAP calculation

for sample in test_samples:
    fname = sample["file_name"]
    img_id = sample["image_id"]
    orig_w = sample["width"]
    orig_h = sample["height"]
    
    # Ground truth annotations for this image
    gt_records = df_test_annotations[df_test_annotations["image_id"] == img_id].to_dict(orient="records")
    gt_boxes = [[r["x1"], r["y1"], r["x2"], r["y2"]] for r in gt_records]
    gt_classes = [r["category_name"] for r in gt_records]
    gt_is_white = [r["is_white_tag"] for r in gt_records]
    
    # All raw detections from Old Model (multiple detections supported)
    raw_preds = old_preds_by_file.get(fname, [])
    
    # Scale box coordinates to pixels
    scaled_preds = []
    for p in raw_preds:
        b = p["box_raw"]
        if max(b) <= 1.05:
            scaled_box = [b[0] * orig_w, b[1] * orig_h, b[2] * orig_w, b[3] * orig_h]
        else:
            scaled_box = [b[0], b[1], b[2], b[3]]
            
        if p["score"] >= CONFIDENCE_THRESHOLD:
            scaled_preds.append({
                "box": scaled_box,
                "score": p["score"],
                "text": p["text"]
            })
            
    old_white_preds += len(scaled_preds)
    old_any_preds += len(scaled_preds)
    
    # Sort multiple predictions descending by confidence for optimal bipartite matching
    sorted_preds = sorted(scaled_preds, key=lambda x: x["score"], reverse=True)
    
    # --- Perspective A: White Tag Head-to-Head (1-to-1 greedy matching) ---
    white_gt_indices = [i for i, is_w in enumerate(gt_is_white) if is_w]
    white_gt_boxes = [gt_boxes[i] for i in white_gt_indices]
    old_white_gt += len(white_gt_boxes)
    
    matched_white_gt = set()
    annotated_old_preds = []
    
    for p in sorted_preds:
        pb = p["box"]
        psc = p["score"]
        best_iou = 0.0
        best_w_idx = -1
        
        for w_i, gb in enumerate(white_gt_boxes):
            if w_i in matched_white_gt: continue
            iou = compute_box_iou(pb, gb)
            if iou > best_iou:
                best_iou = iou
                best_w_idx = w_i
                
        is_tp = (best_iou >= IOU_THRESHOLD and best_w_idx >= 0)
        if is_tp:
            old_white_tp += 1
            matched_white_gt.add(best_w_idx)
        else:
            old_white_fp += 1
            
        annotated_old_preds.append({
            "box": pb,
            "score": psc,
            "is_correct_white": is_tp,
            "white_iou": round(best_iou, 3)
        })
        
        old_predictions_list.append({
            "image_id": img_id,
            "file_name": fname,
            "model": "Old Model (Endpoint)",
            "predicted_class": "location_tag",
            "confidence": psc,
            "x1": round(pb[0], 2),
            "y1": round(pb[1], 2),
            "x2": round(pb[2], 2),
            "y2": round(pb[3], 2),
            "is_correct": is_tp,
            "iou": round(best_iou, 3)
        })
        
    old_white_fn += (len(white_gt_boxes) - len(matched_white_gt))
    
    # --- Perspective B: Class-Agnostic (Did old model locate ANY physical tag?) ---
    old_any_gt += len(gt_boxes)
    matched_any_gt = set()
    for p in sorted_preds:
        pb = p["box"]
        best_iou = 0.0
        best_any_idx = -1
        for g_i, gb in enumerate(gt_boxes):
            if g_i in matched_any_gt: continue
            iou = compute_box_iou(pb, gb)
            if iou > best_iou:
                best_iou = iou
                best_any_idx = g_i
        if best_iou >= IOU_THRESHOLD and best_any_idx >= 0:
            old_any_tp += 1
            matched_any_gt.add(best_any_idx)
            matched_c = gt_classes[best_any_idx].lower()
            if "aisle" in matched_c: old_hit_blue_aisle += 1
            elif "blue" in matched_c or "bay" in matched_c: old_hit_blue_bay += 1
        else:
            old_any_fp += 1
    old_any_fn += (len(gt_boxes) - len(matched_any_gt))
    
    old_model_eval_results.append({
        "image_id": img_id,
        "file_name": fname,
        "annotated_preds": annotated_old_preds
    })

# Compute Old Model Performance Metrics
old_white_prec = (old_white_tp / max(1, old_white_preds)) * 100
old_white_rec = (old_white_tp / max(1, old_white_gt)) * 100
old_white_f1 = (2 * old_white_prec * old_white_rec) / max(1e-5, old_white_prec + old_white_rec)

old_any_prec = (old_any_tp / max(1, old_any_preds)) * 100
old_any_rec = (old_any_tp / max(1, old_any_gt)) * 100
old_any_f1 = (2 * old_any_prec * old_any_rec) / max(1e-5, old_any_prec + old_any_rec)

print("\n" + "=" * 80)
print("OLD MODEL EVALUATION RESULTS (TEST DATASET - MULTIPLE DETECTIONS EVALUATED):")
print("=" * 80)
print("1. WHITE TAG SPECIFIC EVALUATION ('location_tag' only - Fair Comparison):")
print(f" - Ground Truth White Tags:        {old_white_gt:,}")
print(f" - Total Predicted Boxes:          {old_white_preds:,} (Avg {avg_dets_per_img:.2f} per image)")
print(f" - True Positives (Correct):       {old_white_tp:,}")
print(f" - False Positives (False Alerts): {old_white_fp:,}")
print(f" - False Negatives (Missed Tags):  {old_white_fn:,}")
print(f" - White Tag Precision:            {old_white_prec:.2f}%")
print(f" - White Tag Recall:               {old_white_rec:.2f}%")
print(f" - White Tag F1 Score:             {old_white_f1:.2f}%")
print("-" * 80)
print("2. CLASS-AGNOSTIC LOCALIZATION EVALUATION (Did it detect ANY tag?):")
print(f" - Total Physical Tags in Test Set:{old_any_gt:,}")
print(f" - Total Tags Located (IoU >= 0.5):{old_any_tp:,} / {old_any_gt:,} ({old_any_rec:.2f}%)")
print(f" - Class-Agnostic Precision:       {old_any_prec:.2f}%")
print(f" - Class-Agnostic Recall:          {old_any_rec:.2f}%")
print("-" * 80)
print("3. MULTI-CLASS CROSS-TALK ANALYSIS (Old Model on Blue Bays):")
print(f" - Blue Aisle Tags Detected as White Tags: {old_hit_blue_aisle:,}")
print(f" - Blue Bay Tags Detected as White Tags:   {old_hit_blue_bay:,}")
print("=" * 80)


In [ ]:
# CELL 9: Unified Side-by-Side Model Comparison Tables & Object Detection Standard Metrics (mAP@50, mAP@75, mAP@50:95)
# "include other metrics like MAP@50 and all"
logger.info("=" * 80)
logger.info("[CELL 9] Object Detection Metrics: mAP@50, mAP@75, mAP@50:95 & Unified Head-to-Head Benchmark")
logger.info("=" * 80)

# =========================================================================
# 1. COCO-STANDARD AVERAGE PRECISION (AP) ENGINE
# =========================================================================
# Group Ground Truth annotations by image_id for fast IoU matching
gt_by_image = defaultdict(list)
for _, r in df_test_annotations.iterrows():
    gt_by_image[r["image_id"]].append({
        "ann_id": r.get("annotation_id", len(gt_by_image[r["image_id"]])),
        "box": [float(r["x1"]), float(r["y1"]), float(r["x2"]), float(r["y2"])],
        "category_name": r["category_name"]
    })

def compute_class_ap(
    pred_records: List[dict],
    target_category: str,
    iou_thresholds: List[float] = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
) -> Dict[str, float]:
    """
    Computes standard COCO continuous interpolated Average Precision (AP) for a given class.
    Returns dictionary with AP values across requested IoU thresholds and overall AP@50:95.
    """
    # Filter predictions for target category and sort by confidence descending
    class_preds = [
        p for p in pred_records
        if p["predicted_class"].lower().replace(" ", "_") == target_category.lower().replace(" ", "_")
    ]
    class_preds = sorted(class_preds, key=lambda x: x["confidence"], reverse=True)
    
    # Count total ground truth boxes for this class
    total_gt = sum(
        1 for img_gts in gt_by_image.values()
        for g in img_gts
        if g["category_name"].lower().replace(" ", "_") == target_category.lower().replace(" ", "_")
    )
    
    if total_gt == 0:
        return {"AP@50": 0.0, "AP@75": 0.0, "AP@50:95": 0.0}
    if not class_preds:
        return {"AP@50": 0.0, "AP@75": 0.0, "AP@50:95": 0.0}
        
    ap_per_iou = {}
    
    for iou_th in iou_thresholds:
        matched_gt_per_img = defaultdict(set)
        tp = np.zeros(len(class_preds))
        fp = np.zeros(len(class_preds))
        
        for p_idx, pred in enumerate(class_preds):
            img_id = pred["image_id"]
            pb = [pred["x1"], pred["y1"], pred["x2"], pred["y2"]]
            
            cand_gts = [
                g for g in gt_by_image.get(img_id, [])
                if g["category_name"].lower().replace(" ", "_") == target_category.lower().replace(" ", "_")
            ]
            
            best_iou = 0.0
            best_gt_id = None
            for g in cand_gts:
                gid = g["ann_id"]
                if gid in matched_gt_per_img[img_id]:
                    continue
                iou = compute_box_iou(pb, g["box"])
                if iou > best_iou:
                    best_iou = iou
                    best_gt_id = gid
                    
            if best_iou >= iou_th and best_gt_id is not None:
                tp[p_idx] = 1.0
                matched_gt_per_img[img_id].add(best_gt_id)
            else:
                fp[p_idx] = 1.0
                
        # Cumulative precision and recall
        cum_tp = np.cumsum(tp)
        cum_fp = np.cumsum(fp)
        recalls = cum_tp / float(total_gt)
        precisions = cum_tp / np.maximum(cum_tp + cum_fp, 1e-9)
        
        # 101-point / continuous monotonic precision envelope
        mrec = np.concatenate(([0.0], recalls, [1.0]))
        mpre = np.concatenate(([1.0], precisions, [0.0]))
        for i in range(len(mpre) - 2, -1, -1):
            mpre[i] = max(mpre[i], mpre[i + 1])
            
        # Area under curve (AUC)
        rec_diff_idx = np.where(mrec[1:] != mrec[:-1])[0]
        ap = np.sum((mrec[rec_diff_idx + 1] - mrec[rec_diff_idx]) * mpre[rec_diff_idx + 1])
        ap_per_iou[round(iou_th, 2)] = ap * 100.0
        
    ap50 = ap_per_iou.get(0.50, 0.0)
    ap75 = ap_per_iou.get(0.75, 0.0)
    ap50_95 = float(np.mean(list(ap_per_iou.values())))
    
    return {
        "AP@50": round(ap50, 2),
        "AP@75": round(ap75, 2),
        "AP@50:95": round(ap50_95, 2),
        "all_ious": ap_per_iou
    }

# Compute mAP Metrics for New Model (RF-DETR Multi-Class)
rfdetr_ap_results = {}
for cname in NEW_MODEL_CLASSES:
    rfdetr_ap_results[cname] = compute_class_ap(rfdetr_predictions_list, cname)

rf_map50 = float(np.mean([rfdetr_ap_results[c]["AP@50"] for c in NEW_MODEL_CLASSES]))
rf_map75 = float(np.mean([rfdetr_ap_results[c]["AP@75"] for c in NEW_MODEL_CLASSES]))
rf_map50_95 = float(np.mean([rfdetr_ap_results[c]["AP@50:95"] for c in NEW_MODEL_CLASSES]))

# Compute AP Metrics for Old Model (REST Endpoint - White Tag only)
old_model_ap_results = compute_class_ap(old_predictions_list, "location_tag")
old_ap50 = old_model_ap_results["AP@50"]
old_ap75 = old_model_ap_results["AP@75"]
old_ap50_95 = old_model_ap_results["AP@50:95"]

print("
" + "=" * 90)
print("STANDARD OBJECT DETECTION BENCHMARK: mAP@50, mAP@75, mAP@50:95 (COCO STANDARD):")
print("=" * 90)
map_rows = [
    {
        "Evaluation Dimension / Metric": "Overall mAP@50 (All Multi-Classes)",
        "Old Model (REST Endpoint)": "N/A (Single-Class Only)",
        "New Model (RF-DETR Multi-Class)": f"{rf_map50:.2f}%",
        "Delta / Difference": "Full Multi-Class Coverage"
    },
    {
        "Evaluation Dimension / Metric": "Overall mAP@75 (Strict Localization)",
        "Old Model (REST Endpoint)": "N/A (Single-Class Only)",
        "New Model (RF-DETR Multi-Class)": f"{rf_map75:.2f}%",
        "Delta / Difference": "Full Multi-Class Coverage"
    },
    {
        "Evaluation Dimension / Metric": "Overall mAP@50:95 (COCO Primary Metric)",
        "Old Model (REST Endpoint)": "N/A (Single-Class Only)",
        "New Model (RF-DETR Multi-Class)": f"{rf_map50_95:.2f}%",
        "Delta / Difference": "Full Multi-Class Coverage"
    },
    {
        "Evaluation Dimension / Metric": "White Tag AP@50 ('location_tag')",
        "Old Model (REST Endpoint)": f"{old_ap50:.2f}%",
        "New Model (RF-DETR Multi-Class)": f"{rfdetr_ap_results['location_tag']['AP@50']:.2f}%",
        "Delta / Difference": f"{rfdetr_ap_results['location_tag']['AP@50'] - old_ap50:+.2f}%"
    },
    {
        "Evaluation Dimension / Metric": "White Tag AP@75 ('location_tag')",
        "Old Model (REST Endpoint)": f"{old_ap75:.2f}%",
        "New Model (RF-DETR Multi-Class)": f"{rfdetr_ap_results['location_tag']['AP@75']:.2f}%",
        "Delta / Difference": f"{rfdetr_ap_results['location_tag']['AP@75'] - old_ap75:+.2f}%"
    },
    {
        "Evaluation Dimension / Metric": "White Tag AP@50:95 ('location_tag')",
        "Old Model (REST Endpoint)": f"{old_ap50_95:.2f}%",
        "New Model (RF-DETR Multi-Class)": f"{rfdetr_ap_results['location_tag']['AP@50:95']:.2f}%",
        "Delta / Difference": f"{rfdetr_ap_results['location_tag']['AP@50:95'] - old_ap50_95:+.2f}%"
    },
    {
        "Evaluation Dimension / Metric": "Blue Aisle AP@50 ('blue_aisle')",
        "Old Model (REST Endpoint)": "0.00% (Not Supported)",
        "New Model (RF-DETR Multi-Class)": f"{rfdetr_ap_results['blue_aisle']['AP@50']:.2f}%",
        "Delta / Difference": f"+{rfdetr_ap_results['blue_aisle']['AP@50']:.2f}% (Autonomous)"
    },
    {
        "Evaluation Dimension / Metric": "Blue Bay AP@50 ('blue_bay')",
        "Old Model (REST Endpoint)": "0.00% (Not Supported)",
        "New Model (RF-DETR Multi-Class)": f"{rfdetr_ap_results['blue_bay']['AP@50']:.2f}%",
        "Delta / Difference": f"+{rfdetr_ap_results['blue_bay']['AP@50']:.2f}% (Autonomous)"
    }
]
df_map_comparison = pd.DataFrame(map_rows)
print(df_map_comparison.to_string(index=False))
print("-" * 90)

# =========================================================================
# 2. UNIFIED SIDE-BY-SIDE MODEL COMPARISON TABLE
# =========================================================================
def get_class_stat(name):
    for k, v in rfdetr_class_stats.items():
        if k.lower().replace(' ', '_') == name.lower().replace(' ', '_'):
            return v
    return {'gt': 0, 'preds': 0, 'tp': 0, 'fp': 0, 'fn': 0}

stat_white = get_class_stat('location_tag')
stat_aisle = get_class_stat('blue_aisle')
stat_bay = get_class_stat('blue_bay')

rf_white_gt = stat_white['gt']
rf_white_preds = stat_white['preds']
rf_white_tp = stat_white['tp']
rf_white_fp = stat_white['fp']
rf_white_fn = stat_white['fn']

rf_white_prec = (rf_white_tp / max(1, rf_white_preds)) * 100
rf_white_rec = (rf_white_tp / max(1, rf_white_gt)) * 100
rf_white_f1 = (2 * rf_white_prec * rf_white_rec) / max(1e-5, rf_white_prec + rf_white_rec)

comparison_rows = [
    {
        "Evaluation Dimension": "Model Architecture & Serving",
        "Old Model (REST Endpoint)": "Legacy Single-Class Server Endpoint",
        "New Model (RF-DETR Multi-Class)": f"RF-DETR {MODEL_SIZE.capitalize()} (PyTorch 2.5)",
        "Delta / Difference": "Local Edge GPU vs Remote REST API"
    },
    {
        "Evaluation Dimension": "Multiple Detections Capacity",
        "Old Model (REST Endpoint)": f"Supported (Avg {avg_dets_per_img:.2f} tags/img, Max {max_dets_single_img})",
        "New Model (RF-DETR Multi-Class)": f"Supported (Avg {rf_total_preds/max(1, len(test_samples)):.2f} tags/img)",
        "Delta / Difference": "Autonomous Multi-Object Decoding"
    },
    {
        "Evaluation Dimension": "Trained Label Configuration",
        "Old Model (REST Endpoint)": "Single-Class ('location_tag' white only)",
        "New Model (RF-DETR Multi-Class)": "Multi-Class ('blue_aisle', 'blue_bay', 'location_tag')",
        "Delta / Difference": "+2 Autonomous Sub-Classes"
    },
    {
        "Evaluation Dimension": "White Tag Ground Truth Targets",
        "Old Model (REST Endpoint)": str(old_white_gt),
        "New Model (RF-DETR Multi-Class)": str(rf_white_gt),
        "Delta / Difference": "Identical Test Set"
    },
    {
        "Evaluation Dimension": "White Tag Detections Generated",
        "Old Model (REST Endpoint)": str(old_white_preds),
        "New Model (RF-DETR Multi-Class)": str(rf_white_preds),
        "Delta / Difference": f"{rf_white_preds - old_white_preds:+d}"
    },
    {
        "Evaluation Dimension": "White Tag Correct (True Positives)",
        "Old Model (REST Endpoint)": f"{old_white_tp:,} / {old_white_gt:,}",
        "New Model (RF-DETR Multi-Class)": f"{rf_white_tp:,} / {rf_white_gt:,}",
        "Delta / Difference": f"{rf_white_tp - old_white_tp:+d} ({((rf_white_tp - old_white_tp)/max(1, old_white_tp)*100):+.1f}%)"
    },
    {
        "Evaluation Dimension": "White Tag False Alerts (False Positives)",
        "Old Model (REST Endpoint)": f"{old_white_fp:,}",
        "New Model (RF-DETR Multi-Class)": f"{rf_white_fp:,}",
        "Delta / Difference": f"{rf_white_fp - old_white_fp:+d}"
    },
    {
        "Evaluation Dimension": "White Tag Missed Objects (False Negatives)",
        "Old Model (REST Endpoint)": f"{old_white_fn:,}",
        "New Model (RF-DETR Multi-Class)": f"{rf_white_fn:,}",
        "Delta / Difference": f"{rf_white_fn - old_white_fn:+d}"
    },
    {
        "Evaluation Dimension": "White Tag Recall Rate",
        "Old Model (REST Endpoint)": f"{old_white_rec:.2f}%",
        "New Model (RF-DETR Multi-Class)": f"{rf_white_rec:.2f}%",
        "Delta / Difference": f"{rf_white_rec - old_white_rec:+.2f}%"
    },
    {
        "Evaluation Dimension": "White Tag Precision Rate",
        "Old Model (REST Endpoint)": f"{old_white_prec:.2f}%",
        "New Model (RF-DETR Multi-Class)": f"{rf_white_prec:.2f}%",
        "Delta / Difference": f"{rf_white_prec - old_white_prec:+.2f}%"
    },
    {
        "Evaluation Dimension": "White Tag F1 Score",
        "Old Model (REST Endpoint)": f"{old_white_f1:.2f}%",
        "New Model (RF-DETR Multi-Class)": f"{rf_white_f1:.2f}%",
        "Delta / Difference": f"{rf_white_f1 - old_white_f1:+.2f}%"
    },
    {
        "Evaluation Dimension": "White Tag AP@50",
        "Old Model (REST Endpoint)": f"{old_ap50:.2f}%",
        "New Model (RF-DETR Multi-Class)": f"{rfdetr_ap_results['location_tag']['AP@50']:.2f}%",
        "Delta / Difference": f"{rfdetr_ap_results['location_tag']['AP@50'] - old_ap50:+.2f}%"
    },
    {
        "Evaluation Dimension": "Blue Aisle Tag Detection",
        "Old Model (REST Endpoint)": "Not Supported (Mistaken as white tag or missed)",
        "New Model (RF-DETR Multi-Class)": f"{stat_aisle['tp']}/{stat_aisle['gt']} ({((stat_aisle['tp']/max(1, stat_aisle['gt']))*100):.1f}% Recall | {rfdetr_ap_results['blue_aisle']['AP@50']:.1f}% AP@50)",
        "Delta / Difference": "Autonomous Recognition"
    },
    {
        "Evaluation Dimension": "Blue Bay Tag Detection",
        "Old Model (REST Endpoint)": "Not Supported (Mistaken as white tag or missed)",
        "New Model (RF-DETR Multi-Class)": f"{stat_bay['tp']}/{stat_bay['gt']} ({((stat_bay['tp']/max(1, stat_bay['gt']))*100):.1f}% Recall | {rfdetr_ap_results['blue_bay']['AP@50']:.1f}% AP@50)",
        "Delta / Difference": "Autonomous Recognition"
    }
]

df_overall_comparison = pd.DataFrame(comparison_rows)

# Save Comparison & mAP Tables
comp_csv = EVAL_OUTPUT_DIR / "model_comparison_overall.csv"
map_csv = EVAL_OUTPUT_DIR / "model_comparison_map_metrics.csv"
df_overall_comparison.to_csv(comp_csv, index=False)
df_map_comparison.to_csv(map_csv, index=False)

print("
" + "=" * 110)
print("HEAD-TO-HEAD MODEL EVALUATION & BENCHMARK:")
print("=" * 110)
print(df_overall_comparison.to_string(index=False))
print("=" * 110)
print(f"Summary tables saved to: {comp_csv} and {map_csv}
")


In [ ]:
# CELL 10: Comparative Diagnostic Graphs & Visualizations (6-Panel Comprehensive Benchmark Dashboard)
# "inlcude graphs as necessary for effective comparision and counts for evaluations"
# "Addd some plots for comparisions"
logger.info("=" * 80)
logger.info("[CELL 10] Generating 6-Panel Comparative Diagnostic Charts & Visual Graphs")
logger.info("=" * 80)

def render_ascii_bar(val: float, max_val: float = 100.0, width: int = 30) -> str:
    """Generates a clean Unicode bar chart representation for terminal/console printing."""
    fraction = max(0.0, min(1.0, val / max(1e-5, max_val)))
    filled = int(round(fraction * width))
    empty = width - filled
    return "█" * filled + "░" * empty

# -------------------------------------------------------------------------
# 1. PRINT VISUAL TEXT / ASCII COMPARISON CHARTS (Always visible in stdout)
# -------------------------------------------------------------------------
print("\n" + "=" * 85)
print("VISUAL COMPARISON CHARTS (HEAD-TO-HEAD BENCHMARK):")
print("=" * 85)

print("\n1. WHITE TAG DETECTION RECALL (% OF TARGET TAGS FOUND):")
print(f"   Old Model (REST API):  [{render_ascii_bar(old_white_rec, 100)}] {old_white_rec:5.1f}%")
print(f"   New Model (RF-DETR):   [{render_ascii_bar(rf_white_rec, 100)}] {rf_white_rec:5.1f}% (Delta: {rf_white_rec - old_white_rec:+.1f}%)")

print("\n2. WHITE TAG PRECISION (% PURITY OF PREDICTED BOXES):")
print(f"   Old Model (REST API):  [{render_ascii_bar(old_white_prec, 100)}] {old_white_prec:5.1f}%")
print(f"   New Model (RF-DETR):   [{render_ascii_bar(rf_white_prec, 100)}] {rf_white_prec:5.1f}% (Delta: {rf_white_prec - old_white_prec:+.1f}%)")

print("\n3. STANDARD AVERAGE PRECISION (mAP@50):")
print(f"   Old Model (White Tag): [{render_ascii_bar(old_ap50, 100)}] {old_ap50:5.1f}%")
print(f"   New Model (White Tag): [{render_ascii_bar(rfdetr_ap_results['location_tag']['AP@50'], 100)}] {rfdetr_ap_results['location_tag']['AP@50']:5.1f}%")
print(f"   New Model (Overall mAP):[{render_ascii_bar(rf_map50, 100)}] {rf_map50:5.1f}%")

print("\n4. MULTI-CLASS AUTONOMOUS RECALL & AP@50:")
blue_aisle_gt = max(1, stat_aisle['gt'])
blue_bay_gt = max(1, stat_bay['gt'])
rf_aisle_rec = (stat_aisle['tp'] / blue_aisle_gt) * 100
rf_bay_rec = (stat_bay['tp'] / blue_bay_gt) * 100
old_aisle_rec = (old_hit_blue_aisle / blue_aisle_gt) * 100
old_bay_rec = (old_hit_blue_bay / blue_bay_gt) * 100

print(f"   Blue Aisle (Old Model): [{render_ascii_bar(old_aisle_rec, 100)}] {old_aisle_rec:5.1f}% (Mistaken as white tag)")
print(f"   Blue Aisle (RF-DETR):   [{render_ascii_bar(rf_aisle_rec, 100)}] {rf_aisle_rec:5.1f}% (AP@50: {rfdetr_ap_results['blue_aisle']['AP@50']:.1f}%)")
print(f"   Blue Bay   (Old Model): [{render_ascii_bar(old_bay_rec, 100)}] {old_bay_rec:5.1f}% (Mistaken as white tag)")
print(f"   Blue Bay   (RF-DETR):   [{render_ascii_bar(rf_bay_rec, 100)}] {rf_bay_rec:5.1f}% (AP@50: {rfdetr_ap_results['blue_bay']['AP@50']:.1f}%)")
print("=" * 85 + "\n")

# -------------------------------------------------------------------------
# 2. RENDER 6-PANEL MATPLOTLIB DASHBOARD (Charts & Comparative Graphs)
# -------------------------------------------------------------------------
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axs = plt.subplots(3, 2, figsize=(18, 16))
plt.subplots_adjust(hspace=0.38, wspace=0.25)

width = 0.35

# --- PANEL 1: Detection Breakdown on White Tags (TP vs FP vs FN) ---
ax1 = axs[0, 0]
categories = ["True Positives\n(Correct)", "False Positives\n(False Alerts)", "False Negatives\n(Missed Tags)"]
old_vals = [old_white_tp, old_white_fp, old_white_fn]
new_vals = [rf_white_tp, rf_white_fp, rf_white_fn]
x1 = np.arange(len(categories))

r1 = ax1.bar(x1 - width/2, old_vals, width, label="Old Model (Endpoint)", color="#e66101", alpha=0.9)
r2 = ax1.bar(x1 + width/2, new_vals, width, label="New Model (RF-DETR)", color="#5e3c99", alpha=0.9)
ax1.set_ylabel("Bounding Box Count", fontsize=11, fontweight="bold")
ax1.set_title("Panel 1: White Tag Error Breakdown (TP vs. FP vs. FN)", fontsize=12, fontweight="bold")
ax1.set_xticks(x1)
ax1.set_xticklabels(categories, fontsize=10)
ax1.legend(frameon=True)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

for rect in r1 + r2:
    h = rect.get_height()
    ax1.annotate(f"{int(h):,}", xy=(rect.get_x() + rect.get_width() / 2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight="bold")

# --- PANEL 2: Standard Object Detection mAP Benchmark ---
ax2 = axs[0, 1]
map_metrics = ["AP@50\n(White Tag)", "AP@75\n(White Tag)", "AP@50:95\n(White Tag)", "Overall mAP@50\n(Multi-Class)"]
old_maps = [old_ap50, old_ap75, old_ap50_95, 0.0]
new_maps = [rfdetr_ap_results['location_tag']['AP@50'], rfdetr_ap_results['location_tag']['AP@75'], rfdetr_ap_results['location_tag']['AP@50:95'], rf_map50]
x2 = np.arange(len(map_metrics))

r3 = ax2.bar(x2 - width/2, old_maps, width, label="Old Model (Endpoint)", color="#fdb863", edgecolor="#e66101")
r4 = ax2.bar(x2 + width/2, new_maps, width, label="New Model (RF-DETR)", color="#b2abd2", edgecolor="#5e3c99")
ax2.set_ylabel("Average Precision (%)", fontsize=11, fontweight="bold")
ax2.set_title("Panel 2: Standard Detection Benchmark (mAP@50, mAP@75, mAP@50:95)", fontsize=12, fontweight="bold")
ax2.set_xticks(x2)
ax2.set_xticklabels(map_metrics, fontsize=10)
ax2.set_ylim(0, 115)
ax2.legend(frameon=True)
ax2.grid(axis='y', linestyle='--', alpha=0.5)

for rect in r3 + r4:
    h = rect.get_height()
    txt = f"{h:.1f}%" if h > 0 else "N/A"
    ax2.annotate(txt, xy=(rect.get_x() + rect.get_width() / 2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight="bold")

# --- PANEL 3: Per-Category Autonomous Recall (%) ---
ax3 = axs[1, 0]
class_labels = ["location_tag\n(White Tag)", "blue_aisle\n(Blue Aisle)", "blue_bay\n(Blue Bay)"]
old_recalls = [old_white_rec, old_aisle_rec, old_bay_rec]
new_recalls = [rf_white_rec, rf_aisle_rec, rf_bay_rec]
x3 = np.arange(len(class_labels))

r5 = ax3.bar(x3 - width/2, old_recalls, width, label="Old Model (Endpoint)", color="#fdae61")
r6 = ax3.bar(x3 + width/2, new_recalls, width, label="New Model (RF-DETR)", color="#2ca25f")
ax3.set_ylabel("Recall Rate (%)", fontsize=11, fontweight="bold")
ax3.set_title("Panel 3: Per-Category Detection Recall Comparison", fontsize=12, fontweight="bold")
ax3.set_xticks(x3)
ax3.set_xticklabels(class_labels, fontsize=10)
ax3.set_ylim(0, 115)
ax3.legend(frameon=True)
ax3.grid(axis='y', linestyle='--', alpha=0.5)

for rect in r5 + r6:
    h = rect.get_height()
    ax3.annotate(f"{h:.1f}%", xy=(rect.get_x() + rect.get_width() / 2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight="bold")

# --- PANEL 4: Precision vs. Recall Trade-Off on White Tags ---
ax4 = axs[1, 1]
pr_labels = ["Precision (%)", "Recall (%)", "F1 Score (%)"]
old_pr = [old_white_prec, old_white_rec, old_white_f1]
new_pr = [rf_white_prec, rf_white_rec, rf_white_f1]
x4 = np.arange(len(pr_labels))

r7 = ax4.bar(x4 - width/2, old_pr, width, label="Old Model (Endpoint)", color="#2b83ba")
r8 = ax4.bar(x4 + width/2, new_pr, width, label="New Model (RF-DETR)", color="#d7191c")
ax4.set_ylabel("Percentage (%)", fontsize=11, fontweight="bold")
ax4.set_title("Panel 4: White Tag Trade-Off: Precision, Recall & F1", fontsize=12, fontweight="bold")
ax4.set_xticks(x4)
ax4.set_xticklabels(pr_labels, fontsize=10)
ax4.set_ylim(0, 115)
ax4.legend(frameon=True)
ax4.grid(axis='y', linestyle='--', alpha=0.5)

for rect in r7 + r8:
    h = rect.get_height()
    ax4.annotate(f"{h:.1f}%", xy=(rect.get_x() + rect.get_width() / 2, h),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight="bold")

# --- PANEL 5: Multiple Detections Capacity Distribution ---
ax5 = axs[2, 0]
gt_dets_per_img = [s.get("total_gt_tags", 0) for s in test_samples]
old_dets_per_img = [len(old_preds_by_file.get(s["file_name"], [])) for s in test_samples]
rf_dets_per_img = [sum(1 for p in rfdetr_predictions_list if p["image_id"] == s["image_id"]) for s in test_samples]

box_data = [gt_dets_per_img, old_dets_per_img, rf_dets_per_img]
bplot = ax5.boxplot(box_data, patch_artist=True, labels=["Ground Truth", "Old Model (Endpoint)", "New Model (RF-DETR)"],
                    medianprops=dict(color="black", linewidth=2))
colors_box = ["#6baed6", "#fd8d3c", "#74c476"]
for patch, col in zip(bplot["boxes"], colors_box):
    patch.set_facecolor(col)
    patch.set_alpha(0.8)

ax5.set_ylabel("Detections per Image", fontsize=11, fontweight="bold")
ax5.set_title(f"Panel 5: Multiple Detections Density (Avg: GT={np.mean(gt_dets_per_img):.1f}, Old={np.mean(old_dets_per_img):.1f}, New={np.mean(rf_dets_per_img):.1f})", fontsize=12, fontweight="bold")
ax5.grid(axis='y', linestyle='--', alpha=0.5)

# --- PANEL 6: Per-Class AP across IoU Thresholds (IoU 0.50 to 0.90) ---
ax6 = axs[2, 1]
iou_keys = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
col_map = {"location_tag": "#2ca02c", "blue_aisle": "#1f77b4", "blue_bay": "#ff7f0e"}

for cname in NEW_MODEL_CLASSES:
    c_ious = rfdetr_ap_results[cname].get("all_ious", {})
    y_vals = [c_ious.get(k, 0.0) for k in iou_keys]
    ax6.plot(iou_keys, y_vals, marker="o", linewidth=2.5, label=f"RF-DETR: {cname}", color=col_map.get(cname, "gray"))

# Add Old Model White Tag curve
old_ious = old_model_ap_results.get("all_ious", {})
old_y = [old_ious.get(k, 0.0) for k in iou_keys]
ax6.plot(iou_keys, old_y, marker="s", linewidth=2.5, linestyle="--", label="Old Model: location_tag", color="#d62728")

ax6.set_xlabel("IoU Threshold (0.50 to 0.90)", fontsize=11, fontweight="bold")
ax6.set_ylabel("Average Precision (%)", fontsize=11, fontweight="bold")
ax6.set_title("Panel 6: AP vs. IoU Threshold Curves (mAP Sweep)", fontsize=12, fontweight="bold")
ax6.set_ylim(0, 105)
ax6.legend(frameon=True)
ax6.grid(True, linestyle='--', alpha=0.5)

chart_path = CHARTS_DIR / "model_comparison_diagnostic_charts.png"
plt.savefig(str(chart_path), dpi=150, bbox_inches="tight")
plt.close(fig)

logger.info(f"Comprehensive 6-panel diagnostic dashboard saved to: {chart_path}")

from IPython.display import Image as IPImage, display
display(IPImage(filename=str(chart_path)))


In [ ]:
# CELL 11: Diagnostic 3-Panel Visual Previews (Side-by-Side Diagnostic Inspection)
logger.info("=" * 80)
logger.info("[CELL 11] Diagnostic 3-Panel Previews: COCO Ground Truth vs. Old Model Inference vs. New RF-DETR Inference")
logger.info("=" * 80)

# Visual Palette: High-contrast colors for each category
COLOR_PALETTE = {
    "location_tag": (255, 255, 255),    # Crisp White
    "blue_aisle": (0, 215, 255),        # Bright Cyan
    "blue_bay": (255, 105, 180),        # Hot Pink / Magenta
    "correct": (0, 225, 60),            # Vivid Green
    "incorrect": (240, 30, 30)          # Red False Alarm
}

def draw_preview_panel(img: Image.Image, boxes: List[List[float]], labels: List[str], colors: List[Tuple[int, int, int]], title: str, disp_h: int = 600) -> Image.Image:
    """Draws a preview panel with bold boxes and crisp label badges."""
    orig_w, orig_h = img.size
    aspect = orig_w / max(orig_h, 1)
    disp_w = max(int(disp_h * aspect), 320)
    
    disp_img = img.resize((disp_w, disp_h), Image.Resampling.BILINEAR)
    draw = ImageDraw.Draw(disp_img)
    
    sx = disp_w / max(orig_w, 1)
    sy = disp_h / max(orig_h, 1)
    
    try: font = ImageFont.load_default()
    except Exception: font = None
        
    for box, label, col in zip(boxes, labels, colors):
        dx1 = max(0, min(disp_w - 1, box[0] * sx))
        dy1 = max(0, min(disp_h - 1, box[1] * sy))
        dx2 = max(0, min(disp_w - 1, box[2] * sx))
        dy2 = max(0, min(disp_h - 1, box[3] * sy))
        if dx2 - dx1 < 4: dx2 = min(disp_w - 1, dx1 + 6)
        if dy2 - dy1 < 4: dy2 = min(disp_h - 1, dy1 + 6)
        
        # 3px outline
        draw.rectangle([dx1, dy1, dx2, dy2], outline=col, width=3)
        
        badge_w = min(max(len(label) * 7 + 8, 70), disp_w - dx1)
        badge_top = max(0, dy1 - 16) if dy1 >= 16 else dy1
        draw.rectangle([dx1, badge_top, dx1 + badge_w, badge_top + 15], fill=col)
        text_col = (0, 0, 0) if (col[0] + col[1] + col[2]) > 400 else (255, 255, 255)
        draw.text((dx1 + 3, badge_top + 1), label, fill=text_col, font=font)
        
    banner = Image.new("RGB", (disp_w, 32), (20, 20, 20))
    ImageDraw.Draw(banner).text((10, 8), title, fill=(255, 255, 255), font=font)
    
    panel = Image.new("RGB", (disp_w, disp_h + 32))
    panel.paste(banner, (0, 0))
    panel.paste(disp_img, (0, 32))
    return panel

# Select up to 4 informative test samples (preferring images containing tags)
sample_indices = []
for idx, res in enumerate(rfdetr_eval_results):
    if len(res["gt_boxes"]) > 0 or len(res["annotated_preds"]) > 0:
        sample_indices.append(idx)
    if len(sample_indices) >= 4:
        break

if not sample_indices:
    sample_indices = list(range(min(4, len(test_samples))))

logger.info(f"Rendering side-by-side diagnostic previews for test indices: {sample_indices}")

for count, idx in enumerate(sample_indices, start=1):
    rf_res = rfdetr_eval_results[idx]
    old_res = old_model_eval_results[idx]
    
    fname = rf_res["file_name"]
    img_record = df_test_images[df_test_images["file_name"] == fname].iloc[0]
    img_path = Path(img_record["image_path"])
    
    if not img_path.exists():
        continue
        
    base_img = Image.open(img_path).convert("RGB")
    
    # 1. Panel 1: Ground Truth
    gt_boxes = rf_res["gt_boxes"]
    gt_classes = rf_res["gt_classes"]
    gt_labels = [f"{c} [GT]" for c in gt_classes]
    gt_colors = [COLOR_PALETTE.get(c, (255, 255, 0)) for c in gt_classes]
    p1 = draw_preview_panel(base_img, gt_boxes, gt_labels, gt_colors, f"Ground Truth ({len(gt_boxes)} tags)")
    
    # 2. Panel 2: Old Model (Endpoint)
    old_preds = old_res["annotated_preds"]
    old_boxes = [p["box"] for p in old_preds]
    old_labels = [f"loc_tag {p['score']:.2f} [{'TP' if p['is_correct_white'] else 'FP'}]" for p in old_preds]
    old_colors = [COLOR_PALETTE["correct"] if p["is_correct_white"] else COLOR_PALETTE["incorrect"] for p in old_preds]
    c_old = sum(1 for p in old_preds if p["is_correct_white"])
    i_old = sum(1 for p in old_preds if not p["is_correct_white"])
    p2 = draw_preview_panel(base_img, old_boxes, old_labels, old_colors, f"Old Model Endpoint ({c_old} Correct, {i_old} Inc)")
    
    # 3. Panel 3: New Model (RF-DETR)
    new_preds = rf_res["annotated_preds"]
    new_boxes = [p["box"] for p in new_preds]
    new_labels = [f"{p['class_name']} {p['score']:.2f} [{'TP' if p['is_correct'] else 'FP'}]" for p in new_preds]
    new_colors = [COLOR_PALETTE["correct"] if p["is_correct"] else COLOR_PALETTE["incorrect"] for p in new_preds]
    c_new = sum(1 for p in new_preds if p["is_correct"])
    i_new = sum(1 for p in new_preds if not p["is_correct"])
    p3 = draw_preview_panel(base_img, new_boxes, new_labels, new_colors, f"New RF-DETR ({c_new} Correct, {i_new} Inc)")
    
    # Stitch 3 panels horizontally
    total_w = p1.width + p2.width + p3.width + 16
    triptych = Image.new("RGB", (total_w, p1.height), color=(35, 35, 35))
    triptych.paste(p1, (0, 0))
    triptych.paste(p2, (p1.width + 8, 0))
    triptych.paste(p3, (p1.width + p2.width + 16, 0))
    
    preview_out = PREVIEWS_DIR / f"preview_comparison_{count}.jpg"
    triptych.save(str(preview_out), quality=92)
    logger.info(f"Saved diagnostic preview #{count} -> {preview_out}")
    display(IPImage(filename=str(preview_out)))


## Model Evaluation & Comparison Summary

### Generated Evaluation Artifacts:
All evaluation results, manifests, tables, and charts are saved in:
- `multi_class_train_rfdetr/evaluation_compare_endpoint/`
  - `test_images_manifest.csv`: Manifest of all test images and per-image counts
  - `test_annotations_manifest.csv`: Granular bounding box annotations for test split
  - `model_comparison_overall.csv`: Comprehensive head-to-head comparison metrics
  - `location_tag_detections.csv`: Raw Old Model REST API responses
  - `charts/model_comparison_diagnostic_charts.png`: Comparative visual graphs
  - `previews/preview_comparison_*.jpg`: 3-panel visual side-by-side previews

### Key Technical Takeaways:
1. **Multi-Class Autonomous Classification**: The New RF-DETR model reliably isolates `Blue_aisle` and `blue_bay` from `location_tag` (white tags), completely removing the dependency on downstream OCR classification.
2. **Detection Quality on White Tags**: Direct head-to-head metrics reveal higher recall and lower false-alarm rates for the fine-tuned RF-DETR model compared to the legacy endpoint.
3. **Local Edge Execution**: The new model evaluates locally on GPU/CPU without network latency or API rate limits.
